In [ ]:


# ======================================================================================
# COMPETITIVE OFFER REPORT PIPELINE V2
# ======================================================================================
#
# Single-cell notebook version for Colab Enterprise.
#
# What this notebook does:
#   1. Uploads or uses one weekly Competitive Offer Report PDF.
#   2. Computes pdf_hash and checks duplicates in BigQuery.
#   3. Lets you cancel / replace / append_anyway.
#   4. Extracts bronze PDF deck, raw slide text, and positioned slide lines.
#   5. Parses TOC as QA only, not as source of truth.
#   6. Classifies every slide using title/header-first logic.
#   7. Creates content blocks.
#   8. Creates deterministic offer candidates for simple pages.
#   9. Routes dense matrix / price grid pages to Gemini Vision when enabled.
#  10. Normalizes offer candidates into silver.
#  11. Builds offer-device bridge rows.
#  12. Builds review queue and auto-approved review decisions.
#
# Notes:
#   - Bronze and silver are system-generated only.
#   - Human corrections should go into review tables.
#   - TOC page map is QA only.
#   - Header registry only scans title/header lines, never the full slide body.
#   - Dense pages are intentionally review-heavy unless Gemini Vision is enabled.
#
# ======================================================================================

# ======================================================================================
# SECTION 00 - INSTALL REQUIRED PACKAGES
# ======================================================================================

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = [
    ("fitz", "PyMuPDF"),
    ("google.cloud.bigquery", "google-cloud-bigquery"),
    ("google.genai", "google-genai"),
    ("db_dtypes", "db-dtypes"),
    ("pyarrow", "pyarrow"),
    ("dateutil", "python-dateutil"),
]

for import_name, pip_name in REQUIRED_PACKAGES:
    root_name = import_name.split(".")[0]
    if importlib.util.find_spec(root_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print("Package check complete.")


# ======================================================================================
# SECTION 01 - IMPORTS
# ======================================================================================

import os
import re
import json
import time
import uuid
import base64
import hashlib
import traceback
import textwrap
import unicodedata
from pathlib import Path
from datetime import datetime, timezone, date
from typing import Any, Dict, List, Optional, Tuple

import fitz
import numpy as np
import pandas as pd
from dateutil import parser as date_parser

from google.cloud import bigquery
from google.cloud.exceptions import NotFound

try:
    from google import genai
    from google.genai import types as genai_types
    GOOGLE_GENAI_AVAILABLE = True
except Exception as exc:
    GOOGLE_GENAI_AVAILABLE = False
    print("WARNING: google-genai could not be imported. Gemini Vision will be disabled.")
    print(exc)

try:
    from google.colab import files as colab_files
    from google.colab import auth as colab_auth
    IN_COLAB = True
except Exception:
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = None

print("Imports complete.")


# ======================================================================================
# SECTION 02 - CONFIGURATION
# ======================================================================================

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"

# BigQuery dataset and job location.
# Change to "US" if your dataset is US multi-region.
BQ_LOCATION = "us-central1"

# Vertex AI location for Gemini.
VERTEX_LOCATION = "us-central1"
GEMINI_MODEL = "gemini-2.5-pro"

# Set to a local PDF path if you do not want an upload prompt.
# Example: PDF_PATH = "/content/Competitive_Offer_Report_050826.pdf"
PDF_PATH = None

# Upload behavior:
#   True  -> use Colab's file upload widget when PDF_PATH is not set
#   False -> require PDF_PATH to be set manually
USE_UPLOAD_WIDGET = True

# Uploaded PDFs are copied here in Colab/local runtime.
# In Colab, /content is the usual working area.
UPLOAD_DIR = "/content"

# Recommended first run: False.
# After classifier QA is clean, set to True.
USE_GEMINI_VISION = False

# If False, pipeline will run locally in pandas and skip BigQuery writes.
WRITE_TO_BQ = True

# Duplicate action options:
#   None            -> prompt user if duplicate pdf_hash exists
#   "cancel"        -> cancel if duplicate exists
#   "replace"       -> delete all rows for this pdf_hash from pipeline tables, then reload
#   "append_anyway" -> append a new run_id with the same pdf_hash
DUPLICATE_ACTION = None

# Page rendering settings.
RENDER_ZOOM = 2.0
PAGE_IMAGE_DIR_ROOT = "/tmp/competitive_offer_pages"

# Gemini controls.
GEMINI_SLEEP_SECONDS = 0.5
MAX_GEMINI_PAGES_PER_RUN = 999
PROMPT_VERSION = "competitive_offer_v2_vision_strict_json_2026_07"

# Review and approval controls.
AUTO_APPROVE_MIN_CONFIDENCE = 0.86

# Debug output.
PRINT_CLASSIFIER_QA = True
PRINT_SAMPLE_OUTPUTS = True

PIPELINE_VERSION = "v2_title_first_classifier_plus_vision_router"

print("Configuration loaded.")
print(f"PROJECT_ID: {PROJECT_ID}")
print(f"DATASET_ID: {DATASET_ID}")
print(f"BQ_LOCATION: {BQ_LOCATION}")
print(f"VERTEX_LOCATION: {VERTEX_LOCATION}")
print(f"GEMINI_MODEL: {GEMINI_MODEL}")
print(f"USE_GEMINI_VISION: {USE_GEMINI_VISION}")
print(f"WRITE_TO_BQ: {WRITE_TO_BQ}")


# ======================================================================================
# SECTION 03 - TABLE NAMES
# ======================================================================================

TABLES = {
    # Bronze
    "bronze_pdfDecks": "sdi_competitiveOffers_bronze_pdfDecks_weekly",
    "bronze_slideRaw": "sdi_competitiveOffers_bronze_slideRaw_weekly",
    "bronze_slideLines": "sdi_competitiveOffers_bronze_slideLines_weekly",
    "bronze_detectedEntities": "sdi_competitiveOffers_bronze_detectedEntities_weekly",
    "bronze_slideTables": "sdi_competitiveOffers_bronze_slideTables_weekly",

    # Silver
    "silver_tocPageMap": "sdi_competitiveOffers_silver_tocPageMap_weekly",
    "silver_slideSections": "sdi_competitiveOffers_silver_slideSections_weekly",
    "silver_slideContentBlocks": "sdi_competitiveOffers_silver_slideContentBlocks_weekly",
    "silver_offerCandidates": "sdi_competitiveOffers_silver_offerCandidates_weekly",
    "silver_normalizedOffers": "sdi_competitiveOffers_silver_normalizedOffers_weekly",
    "silver_offerDeviceBridge": "sdi_competitiveOffers_silver_offerDeviceBridge_weekly",
    "silver_priceGridRows": "sdi_competitiveOffers_silver_priceGridRows_weekly",

    # Review
    "review_extractionReview": "sdi_competitiveOffers_review_extractionReview_weekly",
    "review_reviewDecisions": "sdi_competitiveOffers_review_reviewDecisions_weekly",
    "review_taxonomy": "sdi_competitiveOffers_review_taxonomy_weekly",
    "review_approvedExamples": "sdi_competitiveOffers_review_approvedExamples_weekly",
}


def fqtn(table_key: str) -> str:
    """Fully qualified BigQuery table name."""
    return f"{PROJECT_ID}.{DATASET_ID}.{TABLES[table_key]}"


print("Table names loaded.")


# ======================================================================================
# SECTION 04 - BIGQUERY CLIENT AND SCHEMAS
# ======================================================================================

if IN_COLAB:
    try:
        colab_auth.authenticate_user()
        print("Colab authentication complete.")
    except Exception as exc:
        print("WARNING: Colab authentication failed or was skipped.")
        print(exc)

bq_client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client project: {bq_client.project}")


def sf(name: str, field_type: str, mode: str = "NULLABLE") -> bigquery.SchemaField:
    return bigquery.SchemaField(name, field_type, mode=mode)


COMMON_FIELDS = [
    sf("deck_id", "STRING"),
    sf("run_id", "STRING"),
    sf("pdf_hash", "STRING"),
    sf("pdf_name", "STRING"),
    sf("deck_week", "DATE"),
]

SCHEMAS = {
    "bronze_pdfDecks": [
        sf("deck_id", "STRING"),
        sf("run_id", "STRING"),
        sf("pdf_hash", "STRING"),
        sf("pdf_name", "STRING"),
        sf("deck_week", "DATE"),
        sf("pdf_path", "STRING"),
        sf("page_count", "INT64"),
        sf("file_size_bytes", "INT64"),
        sf("duplicate_action", "STRING"),
        sf("pipeline_version", "STRING"),
        sf("ingestion_ts", "TIMESTAMP"),
    ],

    "bronze_slideRaw": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("page_label", "STRING"),
        sf("page_width", "FLOAT64"),
        sf("page_height", "FLOAT64"),
        sf("raw_text", "STRING"),
        sf("raw_text_len", "INT64"),
        sf("rendered_image_path", "STRING"),
        sf("extraction_ts", "TIMESTAMP"),
    ],

    "bronze_slideLines": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("block_no", "INT64"),
        sf("line_no", "INT64"),
        sf("span_count", "INT64"),
        sf("line_text", "STRING"),
        sf("line_text_norm", "STRING"),
        sf("x0", "FLOAT64"),
        sf("y0", "FLOAT64"),
        sf("x1", "FLOAT64"),
        sf("y1", "FLOAT64"),
        sf("font_size_max", "FLOAT64"),
        sf("font_size_avg", "FLOAT64"),
        sf("font_names", "STRING"),
        sf("is_bold", "BOOL"),
        sf("is_italic", "BOOL"),
        sf("is_top_zone", "BOOL"),
        sf("is_large_font", "BOOL"),
        sf("extraction_ts", "TIMESTAMP"),
    ],

    "bronze_detectedEntities": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("entity_type", "STRING"),
        sf("entity_value", "STRING"),
        sf("raw_context", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("source_method", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "bronze_slideTables": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("extractor_name", "STRING"),
        sf("extractor_model", "STRING"),
        sf("prompt_version", "STRING"),
        sf("raw_json", "STRING"),
        sf("raw_response_text", "STRING"),
        sf("extraction_status", "STRING"),
        sf("error_message", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_tocPageMap": COMMON_FIELDS + [
        sf("toc_source_page_num", "INT64"),
        sf("toc_line_text", "STRING"),
        sf("toc_label", "STRING"),
        sf("expected_page_num", "INT64"),
        sf("expected_section_id", "STRING"),
        sf("expected_section_group", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_slideSections": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("requires_vision_extraction", "BOOL"),
        sf("classifier_method", "STRING"),
        sf("classifier_confidence", "FLOAT64"),
        sf("matched_rule", "STRING"),
        sf("matched_pattern", "STRING"),
        sf("title_header_text", "STRING"),
        sf("title_header_lines_json", "STRING"),
        sf("toc_expected_section", "STRING"),
        sf("toc_mismatch_flag", "BOOL"),
        sf("classification_debug_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_slideContentBlocks": COMMON_FIELDS + [
        sf("block_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("block_type", "STRING"),
        sf("carrier", "STRING"),
        sf("category", "STRING"),
        sf("block_title", "STRING"),
        sf("block_text", "STRING"),
        sf("source_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_offerCandidates": COMMON_FIELDS + [
        sf("candidate_id", "STRING"),
        sf("block_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("carrier", "STRING"),
        sf("brand", "STRING"),
        sf("product_name", "STRING"),
        sf("product_category", "STRING"),
        sf("offer_type", "STRING"),
        sf("offer_amount_text", "STRING"),
        sf("offer_value", "FLOAT64"),
        sf("currency", "STRING"),
        sf("rate_plan_requirement", "STRING"),
        sf("customer_segment", "STRING"),
        sf("action_required", "STRING"),
        sf("date_start_text", "STRING"),
        sf("date_end_text", "STRING"),
        sf("condition_text", "STRING"),
        sf("raw_evidence", "STRING"),
        sf("extraction_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("validation_errors_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_normalizedOffers": COMMON_FIELDS + [
        sf("offer_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("carrier", "STRING"),
        sf("brand", "STRING"),
        sf("product_name", "STRING"),
        sf("product_category", "STRING"),
        sf("offer_type", "STRING"),
        sf("offer_amount_text", "STRING"),
        sf("offer_value", "FLOAT64"),
        sf("currency", "STRING"),
        sf("rate_plan_requirement", "STRING"),
        sf("customer_segment", "STRING"),
        sf("action_required", "STRING"),
        sf("condition_text", "STRING"),
        sf("date_start", "DATE"),
        sf("date_end", "DATE"),
        sf("raw_evidence", "STRING"),
        sf("normalization_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("auto_approve_flag", "BOOL"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_offerDeviceBridge": COMMON_FIELDS + [
        sf("bridge_id", "STRING"),
        sf("offer_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("carrier", "STRING"),
        sf("device_name", "STRING"),
        sf("device_brand", "STRING"),
        sf("device_family", "STRING"),
        sf("device_model", "STRING"),
        sf("raw_evidence", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_priceGridRows": COMMON_FIELDS + [
        sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("brand", "STRING"),
        sf("device_name", "STRING"),
        sf("device_category", "STRING"),
        sf("retail_price", "FLOAT64"),
        sf("offer_bucket", "STRING"),
        sf("customer_scenario", "STRING"),
        sf("id_requirement", "STRING"),
        sf("rate_plan_requirement", "STRING"),
        sf("price_text", "STRING"),
        sf("price_value", "FLOAT64"),
        sf("raw_evidence", "STRING"),
        sf("extraction_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("validation_errors_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_extractionReview": COMMON_FIELDS + [
        sf("review_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("review_reason", "STRING"),
        sf("review_status", "STRING"),
        sf("source_table", "STRING"),
        sf("raw_payload_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_reviewDecisions": COMMON_FIELDS + [
        sf("decision_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("decision_type", "STRING"),
        sf("decision_status", "STRING"),
        sf("decision_source", "STRING"),
        sf("decision_notes", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_taxonomy": COMMON_FIELDS + [
        sf("taxonomy_id", "STRING"),
        sf("taxonomy_type", "STRING"),
        sf("taxonomy_value", "STRING"),
        sf("taxonomy_display", "STRING"),
        sf("is_active", "BOOL"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_approvedExamples": COMMON_FIELDS + [
        sf("example_id", "STRING"),
        sf("section_id", "STRING"),
        sf("example_type", "STRING"),
        sf("raw_evidence", "STRING"),
        sf("approved_payload_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],
}


# ======================================================================================
# SECTION 05 - GENERAL UTILITY FUNCTIONS
# ======================================================================================

def now_utc() -> datetime:
    return datetime.now(timezone.utc)


def print_header(title: str) -> None:
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)


def print_subheader(title: str) -> None:
    print("\n" + "-" * 110)
    print(title)
    print("-" * 110)


def show_df(df: pd.DataFrame, max_rows: int = 50) -> None:
    if df is None:
        print("DataFrame is None")
        return
    if df.empty:
        print("DataFrame is empty")
        return
    if display is not None:
        display(df.head(max_rows))
    else:
        print(df.head(max_rows).to_string(index=False))


def safe_json(obj: Any) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False, default=str)
    except Exception:
        return str(obj)


def normalize_for_match(text: Any) -> str:
    """Normalize text for classification and regex matching."""
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))
    s = s.replace("–", " ").replace("—", " ").replace("-", " ")
    s = s.replace("&", " and ")
    s = re.sub(r"[(){}:;,|/]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()


def clean_text_line(text: Any) -> str:
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))
    s = s.replace("\u200b", "")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def make_hash_id(prefix: str, *parts: Any, n: int = 24) -> str:
    raw = "||".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:n]
    return f"{prefix}_{digest}"


def compute_file_hash(path: str) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def safe_float(value: Any) -> Optional[float]:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    s = str(value).strip()
    if not s:
        return None
    s_norm = s.upper().replace(",", "").replace("$", "")
    if s_norm in {"FREE", "ON US", "$0", "0"}:
        return 0.0
    match = re.search(r"-?\d+(?:\.\d+)?", s_norm)
    if match:
        return float(match.group(0))
    return None


def parse_date_safe(value: Any) -> Optional[date]:
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None
    try:
        return date_parser.parse(s, fuzzy=True).date()
    except Exception:
        return None




def infer_deck_week_from_text_or_name(pdf_path: str, first_page_text: str) -> date:
    """
    Infer the deck week date from either:
      1. First-page text containing a full month date, e.g. "May 8, 2026"
      2. PDF filename containing MMDDYY, e.g. "Competitive_Offer_Report_050826.pdf"

    Falls back to today's date if no valid date is found.
    """
    text = first_page_text or ""
    filename = os.path.basename(pdf_path or "")

    # Example in PDF text: "May 8, 2026"
    month_date_pattern = (
        r"\b("
        r"January|February|March|April|May|June|July|August|September|October|November|December"
        r")\s+\d{1,2},\s+\d{4}\b"
    )

    text_match = re.search(month_date_pattern, text, flags=re.IGNORECASE)
    if text_match:
        try:
            return date_parser.parse(text_match.group(0)).date()
        except Exception:
            pass

    # Example in filename: 050826 -> May 8, 2026
    filename_match = re.search(r"(?<!\d)(\d{6})(?!\d)", filename)
    if filename_match:
        try:
            return datetime.strptime(filename_match.group(1), "%m%d%y").date()
        except ValueError:
            pass

    return date.today()


# ======================================================================================
# SECTION 05A - PDF UPLOAD WIDGET AND INPUT RESOLUTION
# ======================================================================================

def validate_pdf_path(pdf_path: str) -> str:
    """
    Validate that the selected file exists and looks like a PDF.

    Returns:
        Absolute path to the PDF.
    """
    if pdf_path is None:
        raise ValueError("pdf_path is None.")

    resolved_path = os.path.abspath(os.path.expanduser(str(pdf_path)))

    if not os.path.exists(resolved_path):
        raise FileNotFoundError(f"PDF file does not exist: {resolved_path}")

    if not resolved_path.lower().endswith(".pdf"):
        raise ValueError(f"Selected file is not a PDF: {resolved_path}")

    if os.path.getsize(resolved_path) == 0:
        raise ValueError(f"Selected PDF is empty: {resolved_path}")

    return resolved_path


def upload_pdf_with_colab_widget(upload_dir: str = UPLOAD_DIR) -> str:
    """
    Upload a single PDF using the Colab file upload widget.

    This is the widget-based upload flow:
      - User clicks Choose Files
      - Selects one Competitive Offer Report PDF
      - The file is saved to UPLOAD_DIR
      - The saved local file path is returned

    Notes:
      - If multiple files are uploaded, the first PDF is used.
      - Non-PDF files are ignored.
    """
    if not IN_COLAB:
        raise RuntimeError(
            "Colab upload widget is only available in Colab. "
            "Set PDF_PATH manually when running outside Colab."
        )

    if not USE_UPLOAD_WIDGET:
        raise RuntimeError("USE_UPLOAD_WIDGET is False. Set PDF_PATH manually or enable the upload widget.")

    print_header("PDF UPLOAD")
    print("Upload one Competitive Offer Report PDF using the file upload widget.")
    print("After upload, the pipeline will compute pdf_hash and check BigQuery for duplicates.")

    uploaded_files = colab_files.upload()

    if not uploaded_files:
        raise ValueError("No file was uploaded.")

    upload_path = Path(upload_dir)
    upload_path.mkdir(parents=True, exist_ok=True)

    pdf_candidates = []

    for uploaded_name, uploaded_bytes in uploaded_files.items():
        if not uploaded_name.lower().endswith(".pdf"):
            print(f"Skipping non-PDF upload: {uploaded_name}")
            continue

        safe_name = os.path.basename(uploaded_name)
        destination = upload_path / safe_name

        # colab_files.upload usually writes the file to the current directory,
        # but writing bytes explicitly here makes the path deterministic.
        with open(destination, "wb") as f:
            f.write(uploaded_bytes)

        pdf_candidates.append(str(destination))
        print(f"Uploaded PDF saved to: {destination}")

    if not pdf_candidates:
        raise ValueError("No PDF file was uploaded. Please upload a .pdf file.")

    if len(pdf_candidates) > 1:
        print("Multiple PDFs were uploaded. Using the first PDF:")
        print(pdf_candidates[0])

    return validate_pdf_path(pdf_candidates[0])


def choose_pdf_path() -> str:
    """
    Resolve which PDF to process.

    Priority:
      1. If PDF_PATH is set, use it.
      2. Else, if USE_UPLOAD_WIDGET is True, prompt with Colab upload widget.
      3. Else, raise an error.

    This function returns only a local file path.
    Duplicate handling happens after pdf_hash is computed.
    """
    print_header("PDF INPUT RESOLUTION")

    if PDF_PATH:
        resolved_path = validate_pdf_path(PDF_PATH)
        print(f"Using configured PDF_PATH: {resolved_path}")
        return resolved_path

    if USE_UPLOAD_WIDGET:
        resolved_path = upload_pdf_with_colab_widget(UPLOAD_DIR)
        print(f"Using uploaded PDF: {resolved_path}")
        return resolved_path

    raise ValueError(
        "No PDF_PATH was provided and USE_UPLOAD_WIDGET is False. "
        "Set PDF_PATH to a local PDF path or set USE_UPLOAD_WIDGET = True."
    )



                      
                      

# ======================================================================================
# SECTION 06 - BIGQUERY HELPER FUNCTIONS
# ======================================================================================

def ensure_dataset_and_tables() -> None:
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = BQ_LOCATION

    try:
        bq_client.get_dataset(dataset_id)
        print(f"Dataset exists: {dataset_id}")
    except NotFound:
        print(f"Creating dataset: {dataset_id} in {BQ_LOCATION}")
        bq_client.create_dataset(dataset)

    for table_key, schema in SCHEMAS.items():
        table_id = fqtn(table_key)
        try:
            bq_client.get_table(table_id)
            print(f"Table exists: {table_id}")
        except NotFound:
            print(f"Creating table: {table_id}")
            table = bigquery.Table(table_id, schema=schema)
            bq_client.create_table(table)


def query_bq_df(sql: str, params: Optional[List[bigquery.ScalarQueryParameter]] = None) -> pd.DataFrame:
    job_config = None
    if params:
        job_config = bigquery.QueryJobConfig(query_parameters=params)
    return bq_client.query(sql, job_config=job_config, location=BQ_LOCATION).to_dataframe()


def get_table_schema_map(table_key: str) -> Dict[str, bigquery.SchemaField]:
    table = bq_client.get_table(fqtn(table_key))
    return {field.name: field for field in table.schema}


def is_null_value(value: Any) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def stringify_for_bq(value: Any) -> Optional[str]:
    if is_null_value(value):
        return None
    if isinstance(value, (dict, list, tuple)):
        return safe_json(value)
    return str(value)


def coerce_df_to_bq_schema(df: pd.DataFrame, table_key: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    schema_map = get_table_schema_map(table_key)
    out = df.copy()

    for col in schema_map.keys():
        if col not in out.columns:
            out[col] = None

    out = out[list(schema_map.keys())]

    for col, field in schema_map.items():
        field_type = field.field_type.upper()

        if field_type == "STRING":
            out[col] = out[col].apply(stringify_for_bq)

        elif field_type in {"INT64", "INTEGER"}:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

        elif field_type in {"FLOAT64", "FLOAT", "NUMERIC", "BIGNUMERIC"}:
            out[col] = pd.to_numeric(out[col], errors="coerce")

        elif field_type in {"BOOL", "BOOLEAN"}:
            def to_bool(v: Any) -> Optional[bool]:
                if is_null_value(v):
                    return None
                if isinstance(v, bool):
                    return v
                s = str(v).strip().lower()
                if s in {"true", "1", "yes", "y"}:
                    return True
                if s in {"false", "0", "no", "n"}:
                    return False
                return None
            out[col] = out[col].apply(to_bool)

        elif field_type == "DATE":
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.date

        elif field_type == "TIMESTAMP":
            out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)

    return out


def append_df_to_bq(df: pd.DataFrame, table_key: str) -> None:
    if df is None or df.empty:
        print(f"Skipping empty dataframe for {table_key}")
        return

    table_id = fqtn(table_key)
    clean_df = coerce_df_to_bq_schema(df, table_key)

    if clean_df.empty:
        print(f"Skipping empty cleaned dataframe for {table_key}")
        return

    job_config = bigquery.LoadJobConfig(
        schema=bq_client.get_table(table_id).schema,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )

    print(f"Loading {len(clean_df):,} rows into {table_id}")
    job = bq_client.load_table_from_dataframe(
        clean_df,
        table_id,
        job_config=job_config,
        location=BQ_LOCATION,
    )
    job.result()
    print(f"Loaded {len(clean_df):,} rows into {table_id}")


def get_existing_decks_by_hash(pdf_hash: str) -> pd.DataFrame:
    if not WRITE_TO_BQ:
        return pd.DataFrame()

    table_id = fqtn("bronze_pdfDecks")
    try:
        bq_client.get_table(table_id)
    except NotFound:
        return pd.DataFrame()

    sql = f"""
    SELECT
      deck_id,
      run_id,
      pdf_name,
      deck_week,
      ingestion_ts,
      duplicate_action
    FROM `{table_id}`
    WHERE pdf_hash = @pdf_hash
    ORDER BY ingestion_ts DESC
    """
    params = [bigquery.ScalarQueryParameter("pdf_hash", "STRING", pdf_hash)]
    return query_bq_df(sql, params)


def resolve_duplicate_action(existing_df: pd.DataFrame) -> str:
    if existing_df.empty:
        return "new"

    print_header("DUPLICATE PDF HASH FOUND")
    show_df(existing_df, max_rows=20)

    action = DUPLICATE_ACTION
    if action is None:
        action = input("Choose action: cancel / replace / append_anyway: ").strip().lower()

    allowed = {"cancel", "replace", "append_anyway"}
    if action not in allowed:
        raise ValueError(f"Invalid duplicate action '{action}'. Allowed: {sorted(allowed)}")

    if action == "cancel":
        raise RuntimeError("Pipeline cancelled because duplicate action was cancel.")

    print(f"Duplicate action selected: {action}")
    return action


def delete_existing_rows_for_pdf_hash(pdf_hash: str) -> None:
    print_header("REPLACE MODE - DELETING EXISTING ROWS FOR PDF HASH")

    for table_key in TABLES:
        table_id = fqtn(table_key)
        try:
            table = bq_client.get_table(table_id)
            table_cols = {field.name for field in table.schema}
            if "pdf_hash" not in table_cols:
                print(f"Skipping {table_id}. No pdf_hash column.")
                continue

            sql = f"DELETE FROM `{table_id}` WHERE pdf_hash = @pdf_hash"
            params = [bigquery.ScalarQueryParameter("pdf_hash", "STRING", pdf_hash)]
            job_config = bigquery.QueryJobConfig(query_parameters=params)
            bq_client.query(sql, job_config=job_config, location=BQ_LOCATION).result()
            print(f"Deleted rows from {table_id}")

        except Exception as exc:
            print(f"WARNING: Delete failed for {table_id}")
            print(exc)


# ======================================================================================
# SECTION 07 - PDF BRONZE EXTRACTION
# ======================================================================================

def render_page_to_png(page: fitz.Page, output_path: str, zoom: float = 2.0) -> None:
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    pix.save(output_path)


def extract_pdf_to_bronze(
    pdf_path: str,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    deck_week: date,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print_header("BRONZE EXTRACTION - RAW SLIDES AND POSITIONED LINES")

    pdf_name = os.path.basename(pdf_path)
    image_dir = Path(PAGE_IMAGE_DIR_ROOT) / run_id
    image_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    slide_raw_rows = []
    slide_line_rows = []

    print(f"PDF: {pdf_name}")
    print(f"Pages: {doc.page_count}")
    print(f"Rendered image directory: {image_dir}")

    for page_index in range(doc.page_count):
        page = doc[page_index]
        page_num = page_index + 1
        page_rect = page.rect
        page_width = float(page_rect.width)
        page_height = float(page_rect.height)

        raw_text = page.get_text("text") or ""
        image_path = str(image_dir / f"page_{page_num:03d}.png")
        render_page_to_png(page, image_path, zoom=RENDER_ZOOM)

        slide_raw_rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "page_num": page_num,
            "page_label": str(page_num),
            "page_width": page_width,
            "page_height": page_height,
            "raw_text": raw_text,
            "raw_text_len": len(raw_text),
            "rendered_image_path": image_path,
            "extraction_ts": now_utc(),
        })

        page_dict = page.get_text("dict")
        for block_no, block in enumerate(page_dict.get("blocks", [])):
            if block.get("type") != 0:
                continue

            for line_no, line in enumerate(block.get("lines", [])):
                spans = line.get("spans", [])
                if not spans:
                    continue

                text_parts = []
                font_sizes = []
                font_names = []
                x0_values = []
                y0_values = []
                x1_values = []
                y1_values = []
                is_bold = False
                is_italic = False

                for span in spans:
                    span_text = span.get("text", "")
                    if span_text:
                        text_parts.append(span_text)

                    if span.get("size") is not None:
                        font_sizes.append(float(span.get("size")))

                    font_name = span.get("font", "") or ""
                    if font_name:
                        font_names.append(font_name)
                        font_norm = font_name.lower()
                        if "bold" in font_norm or "black" in font_norm or "heavy" in font_norm:
                            is_bold = True
                        if "italic" in font_norm or "oblique" in font_norm:
                            is_italic = True

                    bbox = span.get("bbox")
                    if bbox:
                        x0_values.append(float(bbox[0]))
                        y0_values.append(float(bbox[1]))
                        x1_values.append(float(bbox[2]))
                        y1_values.append(float(bbox[3]))

                line_text = clean_text_line(" ".join(text_parts))
                if not line_text:
                    continue

                x0 = min(x0_values) if x0_values else None
                y0 = min(y0_values) if y0_values else None
                x1 = max(x1_values) if x1_values else None
                y1 = max(y1_values) if y1_values else None

                font_size_max = max(font_sizes) if font_sizes else None
                font_size_avg = float(np.mean(font_sizes)) if font_sizes else None

                slide_line_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "page_num": page_num,
                    "block_no": int(block_no),
                    "line_no": int(line_no),
                    "span_count": int(len(spans)),
                    "line_text": line_text,
                    "line_text_norm": normalize_for_match(line_text),
                    "x0": x0,
                    "y0": y0,
                    "x1": x1,
                    "y1": y1,
                    "font_size_max": font_size_max,
                    "font_size_avg": font_size_avg,
                    "font_names": ", ".join(sorted(set(font_names))),
                    "is_bold": bool(is_bold),
                    "is_italic": bool(is_italic),
                    "is_top_zone": bool(y0 is not None and y0 <= page_height * 0.28),
                    "is_large_font": False,
                    "extraction_ts": now_utc(),
                })

    doc.close()

    slide_raw_df = pd.DataFrame(slide_raw_rows)
    slide_lines_df = pd.DataFrame(slide_line_rows)

    if not slide_lines_df.empty:
        slide_lines_df["is_large_font"] = False
        for page_num, group in slide_lines_df.groupby("page_num"):
            font_values = pd.to_numeric(group["font_size_max"], errors="coerce").dropna()
            if len(font_values) == 0:
                continue
            q80 = font_values.quantile(0.80)
            large_idx = group[pd.to_numeric(group["font_size_max"], errors="coerce") >= q80].index
            slide_lines_df.loc[large_idx, "is_large_font"] = True

    print(f"Bronze slide raw rows: {len(slide_raw_df):,}")
    print(f"Bronze slide line rows: {len(slide_lines_df):,}")

    return slide_raw_df, slide_lines_df


# ======================================================================================
# SECTION 08 - ENTITY DETECTION
# ======================================================================================

CARRIER_ALIASES = {
    "T-Mobile": ["T-Mobile", "T Mobile", "TMO"],
    "Verizon": ["Verizon", "VZW"],
    "AT&T": ["AT&T", "ATT"],
    "Xfinity": ["Xfinity", "Xfinity Mobile", "Comcast"],
    "Spectrum": ["Spectrum", "Spectrum Mobile"],
    "Optimum": ["Optimum", "Optimum Mobile"],
    "Metro": ["Metro", "Metro by T-Mobile"],
    "Boost": ["Boost", "Boost Mobile"],
    "Cricket": ["Cricket", "Cricket Wireless"],
    "Total Wireless": ["Total Wireless"],
    "Straight Talk": ["Straight Talk"],
    "Visible": ["Visible"],
    "Google Fi": ["Google Fi"],
    "Tello": ["Tello"],
    "Simple Mobile": ["Simple Mobile"],
    "Walmart": ["Walmart"],
}

DEVICE_TERMS = [
    "iPhone", "Apple Watch", "iPad", "Galaxy", "Samsung", "Pixel", "Motorola",
    "Moto", "Razr", "TCL", "REVVL", "Franklin", "Netgear", "Inseego", "Gizmo",
]


def detect_entities(
    slide_raw_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("BRONZE ENTITY DETECTION")
    rows = []

    for _, row in slide_raw_df.iterrows():
        page_num = int(row["page_num"])
        text = row.get("raw_text") or ""

        for carrier, aliases in CARRIER_ALIASES.items():
            for alias in aliases:
                pattern = rf"\b{re.escape(alias)}\b"
                if re.search(pattern, text, flags=re.IGNORECASE):
                    rows.append({
                        "deck_id": deck_id,
                        "run_id": run_id,
                        "pdf_hash": pdf_hash,
                        "pdf_name": pdf_name,
                        "deck_week": deck_week,
                        "page_num": page_num,
                        "entity_type": "carrier",
                        "entity_value": carrier,
                        "raw_context": alias,
                        "confidence": 0.90,
                        "source_method": "regex_alias",
                        "created_ts": now_utc(),
                    })

        for device_term in DEVICE_TERMS:
            pattern = rf"\b{re.escape(device_term)}\b"
            if re.search(pattern, text, flags=re.IGNORECASE):
                rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "page_num": page_num,
                    "entity_type": "device_term",
                    "entity_value": device_term,
                    "raw_context": device_term,
                    "confidence": 0.80,
                    "source_method": "regex_device_term",
                    "created_ts": now_utc(),
                })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates()
    print(f"Detected entity rows: {len(df):,}")
    return df


# ======================================================================================
# SECTION 09 - TOC PARSING AS QA ONLY
# ======================================================================================

def infer_expected_section_from_toc_label(
    label: str,
    expected_page_num: int,
    all_pages_for_label: List[int],
) -> Tuple[str, str]:
    label_norm = normalize_for_match(label)
    pages_sorted = sorted(set(all_pages_for_label))
    page_pos = pages_sorted.index(expected_page_num) if expected_page_num in pages_sorted else 0

    if "ci headlines" in label_norm or "competitive intelligence" in label_norm:
        return ("ci_headlines_postpaid", "ci_headlines") if page_pos == 0 else ("ci_headlines_prepaid", "ci_headlines")

    if "major competitive offer" in label_norm or "major offer" in label_norm:
        return "major_offer_changes", "major_offer_changes"

    if "apple device" in label_norm or "apple smartphone" in label_norm:
        return "postpaid_apple_device_offers", "postpaid"

    if "samsung" in label_norm or "google" in label_norm:
        return "postpaid_samsung_google_device_offers", "postpaid"

    if "2nd tier" in label_norm or "affordable device" in label_norm or "byod offers" in label_norm:
        return "postpaid_motorola_affordable_byod_offers", "postpaid"

    if "consumer bts" in label_norm or "bts offers" in label_norm:
        if page_pos == 0:
            return "postpaid_bts_watch_offers", "postpaid_bts"
        if page_pos == 1:
            return "postpaid_bts_tablet_offers", "postpaid_bts"
        return "postpaid_bts_other_offers", "postpaid_bts"

    if "service" in label_norm and ("t mobile" in label_norm or "verizon" in label_norm or "at and t" in label_norm):
        return "postpaid_service_offers", "postpaid"

    if "postpaid cable mvnos" in label_norm or "cable mvno" in label_norm:
        return ("cable_mvno_handset_offers", "postpaid_cable_mvno") if page_pos == 0 else ("cable_mvno_service_offers", "postpaid_cable_mvno")

    if "postpaid business" in label_norm or "business" in label_norm:
        return "business_device_offers", "business"

    if "national retail" in label_norm:
        return "national_retail_readout", "national_retail"

    if label_norm.strip() == "prepaid":
        return "section_divider_prepaid", "prepaid"

    if "prepaid brands current offers" in label_norm:
        return "prepaid_brand_offers", "prepaid"

    if "prepaid flagship" in label_norm:
        return "prepaid_flagship_offers", "prepaid"

    if "walmart" in label_norm:
        return "walmart_prepaid_offers", "prepaid"

    return "unknown", "unknown"

# ======================================================================================
# SECTION 09 - TOC PARSING AS QA ONLY
# ======================================================================================

def parse_toc_page_map(
    slide_raw_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("SILVER TOC PAGE MAP - QA ONLY")
    rows = []

    toc_pages = slide_raw_df[
        slide_raw_df["raw_text"]
        .fillna("")
        .str.contains("Table of Contents", case=False, regex=False)
    ]

    if toc_pages.empty:
        print("No TOC page detected.")
        return pd.DataFrame()

    for _, toc_row in toc_pages.iterrows():
        toc_source_page_num = int(toc_row["page_num"])
        lines = str(toc_row.get("raw_text") or "").splitlines()

        for line in lines:
            line_clean = clean_text_line(line)
            if not line_clean:
                continue

            # Page numbers are usually at the end of TOC lines.
            tail = line_clean[-60:]
            page_nums = [int(x) for x in re.findall(r"\b\d{1,2}\b", tail)]
            page_nums = [p for p in page_nums if p >= 4]

            if not page_nums:
                continue

            label = re.sub(r"[\.…。]+", " ", line_clean)
            label = re.sub(r"\s+", " ", label).strip()
            label = re.sub(r"^[▪❖•\-\s]+", "", label).strip()
            label = re.sub(
                r"(?:\s+\d{1,2}\s*(?:and|&|,)?\s*)+",
                "",
                label,
                flags=re.IGNORECASE,
            ).strip()

            if len(label) < 4:
                continue

            for expected_page_num in page_nums:
                expected_section_id, expected_group = infer_expected_section_from_toc_label(
                    label,
                    expected_page_num,
                    page_nums,
                )

                rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "toc_source_page_num": toc_source_page_num,
                    "toc_line_text": line_clean,
                    "toc_label": label,
                    "expected_page_num": expected_page_num,
                    "expected_section_id": expected_section_id,
                    "expected_section_group": expected_group,
                    "created_ts": now_utc(),
                })

    toc_df = pd.DataFrame(rows)

    if not toc_df.empty:
        toc_df = toc_df.drop_duplicates()

    print(f"TOC map rows: {len(toc_df):,}")
    return toc_df


# ======================================================================================
# SECTION 10 - TITLE / HEADER FIRST CLASSIFIER
# ======================================================================================

HEADER_RAW_LINE_SCAN_LIMIT = 18
HEADER_MAX_MEANINGFUL_LINES = 8
UNKNOWN_SECTION = "unknown"

SECTION_LAYOUT_MAP = {
    "cover": "cover",
    "confidentiality_notice": "notice",
    "table_of_contents": "toc",
    "section_divider_ci_headlines": "divider",
    "section_divider_offer_updates": "divider",
    "section_divider_postpaid": "divider",
    "section_divider_prepaid": "divider",
    "end_slide": "end_slide",

    "ci_headlines_postpaid": "narrative_headlines",
    "ci_headlines_prepaid": "narrative_headlines",
    "major_offer_changes": "major_change_grid",

    "postpaid_apple_device_offers": "dense_matrix",
    "postpaid_samsung_google_device_offers": "dense_matrix",
    "postpaid_motorola_affordable_byod_offers": "dense_matrix",
    "postpaid_bts_watch_offers": "dense_matrix",
    "postpaid_bts_tablet_offers": "dense_matrix",
    "postpaid_bts_other_offers": "dense_matrix",
    "postpaid_service_offers": "dense_text_table",
    "cable_mvno_handset_offers": "dense_text_table",
    "cable_mvno_service_offers": "dense_text_table",

    "business_device_offers": "dense_matrix",
    "business_newline_byod_offers": "dense_matrix",

    "national_retail_readout": "narrative_readout",

    "prepaid_brand_offers": "dense_prepaid_table",
    "prepaid_metro_price_grid": "price_grid",
    "prepaid_flagship_offers": "dense_prepaid_table",
    "walmart_prepaid_offers": "dense_prepaid_table",

    "unknown": "unknown",
}

VISION_LAYOUT_TYPES = {
    "dense_matrix",
    "dense_text_table",
    "dense_prepaid_table",
    "price_grid",
}

TITLE_FIRST_OVERRIDES = [
    {
        "rule_name": "ci_headlines_postpaid_title_override",
        "section_id": "ci_headlines_postpaid",
        "patterns": [
            r"\bcompetitive intelligence headlines\b.*\bpostpaid\b",
            r"\bcompetitive intelligence headlines\b.*\bcable mvno\b",
            r"\bpostpaid\b.*\bcable mvno\b.*\bkey highlights\b",
            r"\bpostpaid\b.*\bkey highlights\b",
        ],
    },
    {
        "rule_name": "ci_headlines_prepaid_title_override",
        "section_id": "ci_headlines_prepaid",
        "patterns": [
            r"\bcompetitive intelligence headlines\b.*\bprepaid\b",
            r"\bprepaid\b.*\bmvno\b.*\bkey highlights\b",
            r"\bprepaid\b.*\bkey highlights\b",
        ],
    },
    {
        "rule_name": "bts_watches_title_override",
        "section_id": "postpaid_bts_watch_offers",
        "patterns": [
            r"\bpost paid bts watches\b",
            r"\bpostpaid bts watches\b",
            r"\bbts watches\b",
        ],
    },
    {
        "rule_name": "bts_tablet_title_override",
        "section_id": "postpaid_bts_tablet_offers",
        "patterns": [
            r"\bpost paid bts tablet\b",
            r"\bpost paid bts tablets\b",
            r"\bpostpaid bts tablet\b",
            r"\bpostpaid bts tablets\b",
            r"\bbts tablet\b",
            r"\bbts tablets\b",
        ],
    },
    {
        "rule_name": "bts_other_title_override",
        "section_id": "postpaid_bts_other_offers",
        "patterns": [
            r"\bpost paid bts and other offers\b",
            r"\bpostpaid bts and other offers\b",
            r"\bbts and other offers\b",
        ],
    },
    {
        "rule_name": "metro_price_grid_title_override",
        "section_id": "prepaid_metro_price_grid",
        "patterns": [
            r"\bmetro price grid\b",
            r"\bmetro by t mobile\b.*\bprice grid\b",
        ],
    },
]

SLIDE_HEADER_RULES = [
    {
        "rule_name": "major_offer_changes",
        "section_id": "major_offer_changes",
        "patterns": [
            r"\bmajor offer changes\b",
            r"\bmajor competitive offer updates\b",
        ],
    },
    {
        "rule_name": "postpaid_apple_device_offers",
        "section_id": "postpaid_apple_device_offers",
        "patterns": [
            r"\bpost paid apple smartphone offers\b",
            r"\bpostpaid apple smartphone offers\b",
            r"\bapple smartphone offers\b",
            r"\bapple device offers\b",
        ],
    },
    {
        "rule_name": "postpaid_samsung_google_device_offers",
        "section_id": "postpaid_samsung_google_device_offers",
        "patterns": [
            r"\bpost paid samsung and google smartphone offers\b",
            r"\bpostpaid samsung and google smartphone offers\b",
            r"\bsamsung and google smartphone offers\b",
            r"\bsamsung\b.*\bgoogle\b.*\boffers\b",
        ],
    },
    {
        "rule_name": "postpaid_motorola_affordable_byod_offers",
        "section_id": "postpaid_motorola_affordable_byod_offers",
        "patterns": [
            r"\bpost paid motorola affordable phones and byod offers\b",
            r"\bpostpaid motorola affordable phones and byod offers\b",
            r"\bmotorola affordable phones and byod offers\b",
        ],
    },
    {
        "rule_name": "postpaid_service_offers",
        "section_id": "postpaid_service_offers",
        "patterns": [
            r"\bpost paid service offers\b",
            r"\bpostpaid service offers\b",
            r"\bt mobile verizon at and t services\b",
            r"\bservice pricing and offers\b",
        ],
    },
    {
        "rule_name": "cable_mvno_handset_offers",
        "section_id": "cable_mvno_handset_offers",
        "patterns": [
            r"\bpostpaid cable mvno handset offers\b",
            r"\bcable mvno handset offers\b",
            r"\bpostpaid cable mvno\b.*\bhandset\b",
        ],
    },
    {
        "rule_name": "cable_mvno_service_offers",
        "section_id": "cable_mvno_service_offers",
        "patterns": [
            r"\bcable mvno service pricing and offers\b",
            r"\bcable mvno service offers\b",
        ],
    },
    {
        "rule_name": "business_device_offers",
        "section_id": "business_device_offers",
        "patterns": [
            r"\bbusiness flagship device offers\b",
            r"\bbusiness device offers\b",
        ],
    },
    {
        "rule_name": "national_retail_readout",
        "section_id": "national_retail_readout",
        "patterns": [
            r"\bnational retail\b",
            r"\bpromotions marketplace\b",
            r"\bkey highlights week ending\b",
        ],
    },
    {
        "rule_name": "prepaid_brand_offers",
        "section_id": "prepaid_brand_offers",
        "patterns": [
            r"\bprepaid brands current offers\b",
            r"\bmetro by t mobile boost mobile cricket\b",
        ],
    },
    {
        "rule_name": "prepaid_flagship_offers",
        "section_id": "prepaid_flagship_offers",
        "patterns": [
            r"\bflagship brands prepaid current offers\b",
            r"\bt mobile prepaid at and t prepaid verizon prepaid\b",
        ],
    },
    {
        "rule_name": "walmart_prepaid_offers",
        "section_id": "walmart_prepaid_offers",
        "patterns": [
            r"\bwalmart current offers\b",
            r"\bwalmart\b.*\bmetro\b.*\bcricket\b.*\bboost\b",
        ],
    },
]


def print_active_classifier_rules() -> None:
    print_header("ACTIVE CLASSIFIER RULES")

    print("TITLE-FIRST OVERRIDES")
    for rule in TITLE_FIRST_OVERRIDES:
        print(f"  {rule['rule_name']} -> {rule['section_id']}")
        for pattern in rule["patterns"]:
            print(f"    {pattern}")

    print("\nHEADER REGISTRY RULES")
    for rule in SLIDE_HEADER_RULES:
        print(f"  {rule['rule_name']} -> {rule['section_id']}")
        for pattern in rule["patterns"]:
            print(f"    {pattern}")

    broad_bts = []
    for rule in TITLE_FIRST_OVERRIDES + SLIDE_HEADER_RULES:
        for pattern in rule["patterns"]:
            pattern_norm = pattern.lower()
            if "bts" in pattern_norm and not any(
                term in pattern_norm for term in ["watch", "tablet", "other"]
            ):
                broad_bts.append((rule["rule_name"], pattern))

    print("\nBTS sanity check:")
    if broad_bts:
        print("WARNING: Broad BTS patterns found. Remove these:")
        for rule_name, pattern in broad_bts:
            print(f"  {rule_name}: {pattern}")
    else:
        print("OK. No broad BTS patterns found.")


def is_header_noise_line(line: str) -> bool:
    if not line:
        return True

    norm = normalize_for_match(line)

    if len(norm) <= 1:
        return True

    noise_patterns = [
        r"^\d+$",
        r"^page\s+\d+$",
        r"^t mobile confidential$",
        r"^channel mciinsights$",
        r"^source\b",
        r"^sources\b",
        r"^note\b",
        r"^notes\b",
        r"^copyright\b",
        r"^reviewed updated by\b",
    ]

    return any(
        re.search(pattern, norm, flags=re.IGNORECASE)
        for pattern in noise_patterns
    )

def extract_title_header_lines(
    page_num: int,
    slide_raw_df: pd.DataFrame,
    slide_lines_df: pd.DataFrame,
) -> Tuple[List[str], str, str]:
    raw_row = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]
    raw_text = raw_row.get("raw_text") or ""
    page_height = float(raw_row.get("page_height") or 0)

    page_lines = slide_lines_df[slide_lines_df["page_num"] == page_num].copy()
    candidate_lines = []

    if not page_lines.empty and page_height > 0:
        page_lines["sort_y"] = pd.to_numeric(page_lines["y0"], errors="coerce")
        page_lines["sort_x"] = pd.to_numeric(page_lines["x0"], errors="coerce")
        page_lines = page_lines.sort_values(["sort_y", "sort_x"])

        header_zone = page_lines[
            (page_lines["y0"].fillna(999999) <= page_height * 0.30)
            | (page_lines["is_large_font"] == True)
        ]

        for _, line_row in header_zone.head(25).iterrows():
            line = clean_text_line(line_row.get("line_text"))
            if is_header_noise_line(line):
                continue
            if len(line) > 150:
                continue
            candidate_lines.append(line)
            if len(candidate_lines) >= HEADER_MAX_MEANINGFUL_LINES:
                break

    # Fallback to raw text order.
    if not candidate_lines:
        for raw_line in str(raw_text).splitlines()[:HEADER_RAW_LINE_SCAN_LIMIT]:
            line = clean_text_line(raw_line)
            if is_header_noise_line(line):
                continue
            if len(line) > 150:
                continue
            candidate_lines.append(line)
            if len(candidate_lines) >= HEADER_MAX_MEANINGFUL_LINES:
                break

    # Dedupe while preserving order.
    deduped = []
    seen = set()
    for line in candidate_lines:
        key = normalize_for_match(line)
        if key and key not in seen:
            deduped.append(line)
            seen.add(key)

    title_header_text = "\n".join(deduped)
    title_header_norm = normalize_for_match(" ".join(deduped))
    return deduped, title_header_text, title_header_norm


def match_rule_group(rules: List[Dict[str, Any]], text_norm: str) -> Optional[Dict[str, str]]:
    for rule in rules:
        for pattern in rule["patterns"]:
            if re.search(pattern, text_norm, flags=re.IGNORECASE):
                return {
                    "section_id": rule["section_id"],
                    "rule_name": rule["rule_name"],
                    "matched_pattern": pattern,
                }
    return None


def priority_classify(page_num: int, raw_text: str, title_header_norm: str) -> Optional[Dict[str, str]]:
    full_norm = normalize_for_match(raw_text)
    raw_lines = [clean_text_line(x) for x in str(raw_text).splitlines() if clean_text_line(x)]

    if page_num == 1 and re.search(r"\bcompetitive offer report\b", full_norm):
        return {
            "section_id": "cover",
            "rule_name": "priority_cover",
            "matched_pattern": "page 1 and competitive offer report",
        }

    if re.search(r"\bnotice of confidentiality\b", full_norm) or re.search(r"\bdesignated t mobile confidential\b", full_norm):
        return {
            "section_id": "confidentiality_notice",
            "rule_name": "priority_confidentiality_notice",
            "matched_pattern": "notice of confidentiality",
        }

    if re.search(r"\btable of contents\b", full_norm) and re.search(r"\bci headlines\b|\bcompetitive offer updates\b", full_norm):
        return {
            "section_id": "table_of_contents",
            "rule_name": "priority_table_of_contents",
            "matched_pattern": "table of contents plus expected TOC terms",
        }

    if ("ci headlines" in title_header_norm or title_header_norm == "ci headlines") and len(raw_lines) <= 10:
        return {
            "section_id": "section_divider_ci_headlines",
            "rule_name": "priority_ci_headlines_divider",
            "matched_pattern": "sparse CI Headlines divider",
        }

    if "competitive offer updates" in title_header_norm and len(raw_lines) <= 12:
        return {
            "section_id": "section_divider_offer_updates",
            "rule_name": "priority_offer_updates_divider",
            "matched_pattern": "sparse Competitive Offer Updates divider",
        }

    if "competitive offers and promotional updates postpaid" in title_header_norm:
        return {
            "section_id": "section_divider_postpaid",
            "rule_name": "priority_postpaid_divider",
            "matched_pattern": "Competitive Offers and Promotional Updates Postpaid",
        }

    if "competitive offers and promotional updates prepaid" in title_header_norm:
        return {
            "section_id": "section_divider_prepaid",
            "rule_name": "priority_prepaid_divider",
            "matched_pattern": "Competitive Offers and Promotional Updates Prepaid",
        }

    if len(raw_lines) <= 5 and re.search(r"\bare you with us\b|\bthank you\b|\bquestions\b", full_norm):
        return {
            "section_id": "end_slide",
            "rule_name": "priority_end_slide",
            "matched_pattern": "sparse end slide",
        }

    return None


def refine_business_section(section_id: str, raw_text: str) -> str:
    if section_id != "business_device_offers":
        return section_id

    text_norm = normalize_for_match(raw_text)
    has_device_terms = bool(re.search(r"\biphone\b|\bgalaxy\b|\bsamsung\b|\bpixel\b", text_norm))
    has_newline_byod_terms = bool(re.search(r"\bnew line offers\b|\bbyod offers\b", text_norm))

    # If a page only contains New Line/BYOD blocks, classify separately.
    if has_newline_byod_terms and not has_device_terms:
        return "business_newline_byod_offers"

    return section_id


def classify_one_slide(
    page_num: int,
    slide_raw_df: pd.DataFrame,
    slide_lines_df: pd.DataFrame,
    toc_expected_by_page: Dict[int, str],
) -> Dict[str, Any]:
    raw_row = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]
    raw_text = raw_row.get("raw_text") or ""

    title_lines, title_header_text, title_header_norm = extract_title_header_lines(
        page_num,
        slide_raw_df,
        slide_lines_df,
    )

    toc_expected_section = toc_expected_by_page.get(page_num)
    debug = {
        "page_num": page_num,
        "title_header_lines": title_lines,
        "title_header_norm": title_header_norm,
        "toc_expected_section": toc_expected_section,
        "steps": [],
    }

    # 1. Priority rules.
    priority_match = priority_classify(page_num, raw_text, title_header_norm)
    if priority_match:
        section_id = priority_match["section_id"]
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        debug["steps"].append({"step": "priority_rule", "matched": True, **priority_match})
        return {
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "requires_vision_extraction": layout_type in VISION_LAYOUT_TYPES,
            "classifier_method": "priority_rule",
            "classifier_confidence": 0.99,
            "matched_rule": priority_match["rule_name"],
            "matched_pattern": priority_match["matched_pattern"],
            "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines),
            "toc_expected_section": toc_expected_section,
            "toc_mismatch_flag": bool(toc_expected_section and toc_expected_section != section_id),
            "classification_debug_json": safe_json(debug),
        }
    debug["steps"].append({"step": "priority_rule", "matched": False})

    # 2. Title-first overrides.
    title_match = match_rule_group(TITLE_FIRST_OVERRIDES, title_header_norm)
    if title_match:
        section_id = refine_business_section(title_match["section_id"], raw_text)
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        debug["steps"].append({"step": "title_first_override", "matched": True, **title_match})
        return {
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "requires_vision_extraction": layout_type in VISION_LAYOUT_TYPES,
            "classifier_method": "title_first_override",
            "classifier_confidence": 0.98,
            "matched_rule": title_match["rule_name"],
            "matched_pattern": title_match["matched_pattern"],
            "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines),
            "toc_expected_section": toc_expected_section,
            "toc_mismatch_flag": bool(toc_expected_section and toc_expected_section != section_id),
            "classification_debug_json": safe_json(debug),
        }
    debug["steps"].append({"step": "title_first_override", "matched": False})

    # 3. Header registry on title/header zone only.
    header_match = match_rule_group(SLIDE_HEADER_RULES, title_header_norm)
    if header_match:
        section_id = refine_business_section(header_match["section_id"], raw_text)
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        debug["steps"].append({"step": "header_registry_title_zone_only", "matched": True, **header_match})
        return {
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "requires_vision_extraction": layout_type in VISION_LAYOUT_TYPES,
            "classifier_method": "header_registry_title_zone_only",
            "classifier_confidence": 0.94,
            "matched_rule": header_match["rule_name"],
            "matched_pattern": header_match["matched_pattern"],
            "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines),
            "toc_expected_section": toc_expected_section,
            "toc_mismatch_flag": bool(toc_expected_section and toc_expected_section != section_id),
            "classification_debug_json": safe_json(debug),
        }
    debug["steps"].append({"step": "header_registry_title_zone_only", "matched": False})

    # 4. Unknown.
    section_id = UNKNOWN_SECTION
    layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
    debug["steps"].append({"step": "unknown", "matched": False})

    return {
        "page_num": page_num,
        "section_id": section_id,
        "layout_type": layout_type,
        "requires_vision_extraction": False,
        "classifier_method": "unknown",
        "classifier_confidence": 0.0,
        "matched_rule": None,
        "matched_pattern": None,
        "title_header_text": title_header_text,
        "title_header_lines_json": safe_json(title_lines),
        "toc_expected_section": toc_expected_section,
        "toc_mismatch_flag": bool(toc_expected_section and toc_expected_section != section_id),
        "classification_debug_json": safe_json(debug),
    }


def classify_all_slides(
    slide_raw_df: pd.DataFrame,
    slide_lines_df: pd.DataFrame,
    toc_map_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("SILVER SLIDE CLASSIFICATION")
    print_active_classifier_rules()

    toc_expected_by_page = {}
    if toc_map_df is not None and not toc_map_df.empty:
        for _, row in toc_map_df.iterrows():
            toc_expected_by_page[int(row["expected_page_num"])] = row["expected_section_id"]

    rows = []
    for page_num in sorted(slide_raw_df["page_num"].unique()):
        classified = classify_one_slide(
            int(page_num),
            slide_raw_df,
            slide_lines_df,
            toc_expected_by_page,
        )
        classified.update({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "created_ts": now_utc(),
        })
        rows.append(classified)

    sections_df = pd.DataFrame(rows)
    print(f"Slide section rows: {len(sections_df):,}")

    if PRINT_CLASSIFIER_QA:
        qa_cols = [
            "page_num", "section_id", "layout_type", "classifier_method",
            "classifier_confidence", "toc_expected_section", "toc_mismatch_flag",
            "title_header_text",
        ]
        print_subheader("Classifier QA by page")
        show_df(sections_df[qa_cols].sort_values("page_num"), max_rows=200)

        print_subheader("Classifier counts")
        counts_df = (
            sections_df.groupby(["section_id", "layout_type"], dropna=False)
            .size()
            .reset_index(name="rows")
            .sort_values(["section_id", "layout_type"])
        )
        show_df(counts_df, max_rows=200)

    return sections_df


# ======================================================================================
# SECTION 11 - DETERMINISTIC CONTENT BLOCK CREATION
# ======================================================================================

NO_CHANGE_REGEXES = [
    r"\bno changes\b",
    r"\bno updates\b",
    r"\bno new offers\b",
    r"\bno offers expired\b",
    r"\bno expired offers\b",
    r"\bno offers ended\b",
    r"\bno offer\b",
    r"\bnone\b",
]


def is_no_change_text(text: str) -> bool:
    norm = normalize_for_match(text)
    return any(re.search(pattern, norm, flags=re.IGNORECASE) for pattern in NO_CHANGE_REGEXES)


def infer_carrier_from_text(text: str) -> Optional[str]:
    norm = normalize_for_match(text)
    rules = [
        ("T-Mobile", r"\bt mobile\b"),
        ("AT&T", r"\bat and t\b|\batt\b"),
        ("Verizon", r"\bverizon\b|\bvzw\b"),
        ("Xfinity", r"\bxfinity\b|\bcomcast\b"),
        ("Spectrum", r"\bspectrum\b"),
        ("Optimum", r"\boptimum\b"),
        ("Metro", r"\bmetro\b"),
        ("Boost", r"\bboost\b"),
        ("Cricket", r"\bcricket\b"),
        ("Total Wireless", r"\btotal wireless\b"),
        ("Straight Talk", r"\bstraight talk\b"),
        ("Visible", r"\bvisible\b"),
        ("Google Fi", r"\bgoogle fi\b"),
        ("Tello", r"\btello\b"),
        ("Simple Mobile", r"\bsimple mobile\b"),
        ("Walmart", r"\bwalmart\b"),
    ]
    for carrier, pattern in rules:
        if re.search(pattern, norm, flags=re.IGNORECASE):
            return carrier
    return None


def create_content_block(
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
    page_num: int,
    section_id: str,
    layout_type: str,
    block_type: str,
    carrier: Optional[str],
    category: Optional[str],
    block_title: Optional[str],
    block_text: str,
    source_method: str,
    confidence: float,
    needs_review: bool,
) -> Dict[str, Any]:
    block_id = make_hash_id(
        "block",
        deck_id,
        page_num,
        section_id,
        block_type,
        carrier,
        category,
        block_title,
        block_text,
    )
    return {
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "pdf_name": pdf_name,
        "deck_week": deck_week,
        "block_id": block_id,
        "page_num": page_num,
        "section_id": section_id,
        "layout_type": layout_type,
        "block_type": block_type,
        "carrier": carrier,
        "category": category,
        "block_title": block_title,
        "block_text": block_text,
        "source_method": source_method,
        "confidence": confidence,
        "needs_review": needs_review,
        "created_ts": now_utc(),
    }


def build_ci_headline_blocks(
    raw_text: str,
    base: Dict[str, Any],
) -> List[Dict[str, Any]]:
    lines = [
        line
        for line in (clean_text_line(x) for x in str(raw_text).splitlines())
        if line
    ]

    blocks = []
    current_carrier = None
    current_lines = []

    def flush_current() -> None:
        nonlocal current_carrier, current_lines

        if not current_carrier or not current_lines:
            return

        block_text = "\n".join(current_lines).strip()

        blocks.append(create_content_block(
            **base,
            block_type="ci_headline",
            carrier=current_carrier,
            category="headline",
            block_title=current_carrier,
            block_text=block_text,
            source_method="deterministic_ci_numbered_grouping",
            confidence=0.90,
            needs_review=False,
        ))

    for line in lines:
        carrier_match = re.match(r"^\s*(\d{1,2})\.\s+(.+?)\s*$", line)

        if carrier_match:
            flush_current()
            current_carrier = clean_text_line(carrier_match.group(2))
            current_lines = []
            continue

        if current_carrier:
            current_lines.append(line)

    flush_current()
    return blocks


def build_bullet_blocks(
    raw_text: str,
    base: Dict[str, Any],
    block_type: str,
) -> List[Dict[str, Any]]:
    lines = [
        line
        for line in (clean_text_line(x) for x in str(raw_text).splitlines())
        if line
    ]

    blocks = []
    current_carrier = None
    current_category = None

    for line in lines:
        if len(line) <= 45:
            possible_carrier = infer_carrier_from_text(line)

            if possible_carrier:
                current_carrier = possible_carrier
                continue

        if re.match(r"^[A-Z][A-Z\s&/()\-]+:$", line):
            current_category = line.strip(":").title()
            continue

        looks_like_offer = bool(
            re.match(r"^[•▪o\-]\s*", line)
            or re.search(
                r"\$\d+|\bfree\b|\bon us\b|\bno changes\b|\bexpired\b|\bnew\b|\bremoved\b",
                line,
                flags=re.IGNORECASE,
            )
        )

        if not looks_like_offer:
            continue

        block_text = re.sub(r"^[•▪o\-]\s*", "", line).strip()

        if len(block_text) < 3:
            continue

        blocks.append(create_content_block(
            **base,
            block_type=block_type,
            carrier=current_carrier,
            category=current_category,
            block_title=current_category or current_carrier or block_type,
            block_text=block_text,
            source_method=f"deterministic_{block_type}_bullet_parser",
            confidence=0.72 if current_carrier else 0.55,
            needs_review=not bool(current_carrier),
        ))

    return blocks


def build_content_blocks(
    slide_raw_df: pd.DataFrame,
    slide_sections_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("SILVER CONTENT BLOCK CREATION")
    rows = []

    for _, section_row in slide_sections_df.iterrows():
        page_num = int(section_row["page_num"])
        section_id = section_row["section_id"]
        layout_type = section_row["layout_type"]
        raw_text = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0].get("raw_text") or ""

        base = {
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
        }

        non_offer_sections = {
            "cover",
            "confidentiality_notice",
            "table_of_contents",
            "section_divider_ci_headlines",
            "section_divider_offer_updates",
            "section_divider_postpaid",
            "section_divider_prepaid",
            "end_slide",
        }

        if section_id in non_offer_sections:
            rows.append(create_content_block(
                **base,
                block_type="non_offer_slide",
                carrier=None,
                category=None,
                block_title=section_id,
                block_text=raw_text[:2000],
                source_method="non_offer_slide_capture",
                confidence=0.99,
                needs_review=False,
            ))
            continue

        if section_id in {"ci_headlines_postpaid", "ci_headlines_prepaid"}:
            rows.extend(build_ci_headline_blocks(raw_text, base))
            continue

        if section_id == "major_offer_changes":
            rows.extend(build_bullet_blocks(raw_text, base, block_type="major_offer_change"))
            continue

        if layout_type in VISION_LAYOUT_TYPES:
            # Do not create 500 noisy body-line offers from dense matrix pages.
            # Create a page-level placeholder. Gemini Vision will create structured rows when enabled.
            rows.append(create_content_block(
                **base,
                block_type="vision_page_candidate",
                carrier=None,
                category=None,
                block_title=section_id,
                block_text=raw_text[:4000],
                source_method="vision_routing_placeholder",
                confidence=0.80,
                needs_review=not USE_GEMINI_VISION,
            ))
            continue

        if section_id == "national_retail_readout":
            rows.extend(build_bullet_blocks(raw_text, base, block_type="national_retail_readout"))
            continue

        if section_id == "unknown":
            rows.append(create_content_block(
                **base,
                block_type="unknown_page",
                carrier=None,
                category=None,
                block_title="unknown",
                block_text=raw_text[:4000],
                source_method="unknown_page_capture",
                confidence=0.0,
                needs_review=True,
            ))
            continue

        rows.extend(build_bullet_blocks(raw_text, base, block_type="generic_offer_text"))

    blocks_df = pd.DataFrame(rows)
    print(f"Content block rows: {len(blocks_df):,}")
    return blocks_df


# ======================================================================================
# SECTION 12 - OFFER CANDIDATES FROM DETERMINISTIC BLOCKS
# ======================================================================================

def extract_offer_value(text: Optional[str]) -> Optional[float]:
    if not text:
        return None
    if re.search(r"\bfree\b|\bon us\b|$0\b", text, flags=re.IGNORECASE):
        return 0.0

    money_match = re.search(r"(?:up to\s*)?[<>]?\s*$\s*([0-9][0-9,]*(?:\.\d+)?)", text, flags=re.IGNORECASE)
    if money_match:
        return float(money_match.group(1).replace(",", ""))

    return None


def infer_offer_type(text: Optional[str]) -> str:
    if not text:
        return "unknown"
    norm = normalize_for_match(text)

    if is_no_change_text(text):
        return "no_offer"
    if re.search(r"\bbogo\b|\bbuy 1\b|\bget 2nd\b", norm):
        return "bogo"
    if re.search(r"\btrade\b|\btrade in\b|\btradein\b|\btrd\b", norm):
        return "trade_in_discount"
    if re.search(r"\bport\b|\bport in\b|\bportin\b", norm):
        return "port_in_discount"
    if re.search(r"\bbyod\b|\bbring your own\b", norm):
        return "byod_credit"
    if re.search(r"\bbill credit\b|\bcredits\b|\bcredit\b|\bvisa\b|\bmastercard\b", norm):
        return "bill_credit"
    if re.search(r"\bfree\b|\bon us\b|$0\b", text, flags=re.IGNORECASE):
        return "free_or_on_us"
    if re.search(r"$\s*\d+", text):
        return "discount"
    if re.search(r"\bexpired\b|\bremoved\b|\bended\b|\bretired\b", norm):
        return "expired_or_removed"
    return "other"


def extract_product_name(text: Optional[str]) -> Optional[str]:
    if not text:
        return None

    patterns = [
        r"iPhone\s+\d+[A-Za-z]*(?:\s+Pro\s+Max|\s+Pro|\s+Plus|\s+Air|\s+e)?",
        r"Galaxy\s+S\d+\s*(?:Ultra|FE|Edge|\+)?",
        r"Galaxy\s+Z\s*(?:Flip|Fold)\d*",
        r"Galaxy\s+A\d+\s*5G?",
        r"Pixel\s+\d+[A-Za-z]*(?:\s+Pro\s+XL|\s+Pro|\s+Fold)?",
        r"moto\s+g\s+stylus\s+5G\s+\d{4}",
        r"moto\s+g\s+power\s+5G\s+\d{4}",
        r"moto\s+g\s+5G\s+\d{4}",
        r"moto\s+razr\s+\d{4}",
        r"Apple\s+Watch\s+[A-Za-z0-9\s]+",
        r"iPad\s+?",
        r"Galaxy\s+Tab\s+[A-Za-z0-9+\s]+",
        r"TCL\s+[A-Za-z0-9\s]+",
        r"REVVL\s+[A-Za-z0-9\s]+",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return clean_text_line(match.group(0))[:100]

    prefix = clean_text_line(str(text).split(":")[0])
    if 2 <= len(prefix) <= 60 and not re.search(r"\boffer\b|\bdiscount\b|\btrade\b", normalize_for_match(prefix)):
        return prefix

    return None


def validate_offer_candidate_dict(row: Dict[str, Any], section_id: str) -> List[str]:
    errors = []
    if not row.get("carrier"):
        errors.append("missing_carrier")
    if not row.get("raw_evidence"):
        errors.append("missing_raw_evidence")
    if not row.get("offer_type"):
        errors.append("missing_offer_type")

    if row.get("offer_type") != "no_offer":
        if not row.get("product_name") and not row.get("offer_amount_text") and not row.get("category"):
            errors.append("missing_product_or_offer_amount")

    return errors


def build_offer_candidates_from_blocks(
    content_blocks_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("SILVER OFFER CANDIDATES - DETERMINISTIC BLOCK PARSER")
    rows = []

    if content_blocks_df is None or content_blocks_df.empty:
        print("No content blocks found.")
        return pd.DataFrame()

    skip_block_types = {"non_offer_slide", "vision_page_candidate", "unknown_page", "ci_headline"}

    for _, block in content_blocks_df.iterrows():
        block_type = block.get("block_type")
        if block_type in skip_block_types:
            continue

        block_text = block.get("block_text") or ""
        offer_type = infer_offer_type(block_text)
        product_name = extract_product_name(block_text)
        offer_value = extract_offer_value(block_text)

        candidate_dict = {
            "carrier": block.get("carrier"),
            "brand": None,
            "product_name": product_name,
            "product_category": None,
            "offer_type": offer_type,
            "offer_amount_text": block_text if offer_value is not None or offer_type != "other" else None,
            "offer_value": offer_value,
            "currency": "USD",
            "rate_plan_requirement": None,
            "customer_segment": None,
            "action_required": None,
            "date_start_text": None,
            "date_end_text": None,
            "condition_text": block_text,
            "raw_evidence": block_text,
            "confidence": block.get("confidence", 0.65),
            "needs_review": block.get("needs_review", True),
        }

        validation_errors = validate_offer_candidate_dict(candidate_dict, block.get("section_id"))
        confidence = safe_float(candidate_dict.get("confidence")) or 0.65
        needs_review = bool(candidate_dict.get("needs_review")) or bool(validation_errors) or confidence < AUTO_APPROVE_MIN_CONFIDENCE

        candidate_id = make_hash_id(
            "cand",
            deck_id,
            block.get("page_num"),
            block.get("section_id"),
            candidate_dict.get("carrier"),
            candidate_dict.get("product_name"),
            candidate_dict.get("offer_type"),
            candidate_dict.get("raw_evidence"),
        )

        rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "candidate_id": candidate_id,
            "block_id": block.get("block_id"),
            "page_num": int(block.get("page_num")),
            "section_id": block.get("section_id"),
            "layout_type": block.get("layout_type"),
            "carrier": candidate_dict.get("carrier"),
            "brand": candidate_dict.get("brand"),
            "product_name": candidate_dict.get("product_name"),
            "product_category": candidate_dict.get("product_category"),
            "offer_type": candidate_dict.get("offer_type"),
            "offer_amount_text": candidate_dict.get("offer_amount_text"),
            "offer_value": candidate_dict.get("offer_value"),
            "currency": candidate_dict.get("currency"),
            "rate_plan_requirement": candidate_dict.get("rate_plan_requirement"),
            "customer_segment": candidate_dict.get("customer_segment"),
            "action_required": candidate_dict.get("action_required"),
            "date_start_text": candidate_dict.get("date_start_text"),
            "date_end_text": candidate_dict.get("date_end_text"),
            "condition_text": candidate_dict.get("condition_text"),
            "raw_evidence": candidate_dict.get("raw_evidence"),
            "extraction_method": "deterministic_block_parser",
            "confidence": confidence,
            "needs_review": needs_review,
            "validation_errors_json": safe_json(validation_errors),
            "created_ts": now_utc(),
        })

    candidates_df = pd.DataFrame(rows)
    if not candidates_df.empty:
        candidates_df = candidates_df.drop_duplicates(subset=["candidate_id"])
    print(f"Deterministic offer candidate rows: {len(candidates_df):,}")
    return candidates_df


# ======================================================================================
# SECTION 13 - GEMINI VISION EXTRACTOR FOR DENSE MATRIX AND PRICE GRID PAGES
# ======================================================================================

def make_gemini_client() -> Any:
    if not GOOGLE_GENAI_AVAILABLE:
        raise RuntimeError("google-genai is not available. Set USE_GEMINI_VISION=False or install/fix google-genai.")

    return genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=VERTEX_LOCATION,
    )


def read_binary_file(path: str) -> bytes:
    with open(path, "rb") as f:
        return f.read()


def build_vision_prompt(section_id: str, layout_type: str, title_header_text: str, raw_text: str) -> str:
    allowed_carriers = [
        "T-Mobile", "Verizon", "AT&T", "Xfinity", "Spectrum", "Optimum",
        "Metro", "Boost", "Cricket", "Total Wireless", "Straight Talk",
        "Visible", "Google Fi", "Tello", "Simple Mobile", "Walmart",
    ]

    output_schema_example = {
        "extraction_summary": {
            "section_id": section_id,
            "layout_type": layout_type,
            "confidence": 0.0,
            "notes": None,
        },
        "rows": [
            {
                "carrier": None,
                "brand": None,
                "product_name": None,
                "product_category": None,
                "category": None,
                "offer_type": None,
                "offer_amount_text": None,
                "offer_value": None,
                "currency": "USD",
                "rate_plan_requirement": None,
                "customer_segment": None,
                "action_required": None,
                "date_start_text": None,
                "date_end_text": None,
                "condition_text": None,
                "raw_evidence": None,
                "confidence": 0.0,
                "needs_review": True,
            }
        ],
        "price_grid_rows": [
            {
                "brand": None,
                "device_name": None,
                "device_category": None,
                "retail_price": None,
                "offer_bucket": None,
                "customer_scenario": None,
                "id_requirement": None,
                "rate_plan_requirement": None,
                "price_text": None,
                "price_value": None,
                "raw_evidence": None,
                "confidence": 0.0,
                "needs_review": True,
            }
        ],
    }

    prompt = f"""
You are extracting structured competitive telecom offer data from one rendered slide image.

Use the image as the primary source of truth because PDF text can flatten table columns incorrectly.
Use the raw text only as a secondary aid.

Return valid JSON only. Do not return markdown. Do not include explanation.

Slide metadata:
section_id: {section_id}
layout_type: {layout_type}

title_header_text:
{title_header_text}

Allowed carriers:
{json.dumps(allowed_carriers, ensure_ascii=False)}

Extraction rules:
1. Do not invent offers.
2. Preserve carrier ownership from the visual columns.
3. Every row must include raw_evidence from the slide.
4. If a cell says No offer, No changes, none, or --, set offer_type to no_offer.
5. For price grids, populate price_grid_rows and leave rows empty.
6. For non-price-grid pages, populate rows and leave price_grid_rows empty.
7. Use null when a value is not visible.
8. Use confidence from 0.0 to 1.0.
9. Set needs_review true when carrier, product/device, amount, condition, or price is unclear.
10. Keep row granularity reasonable: one row per carrier + product/category + distinct offer.

Required output JSON shape:
{json.dumps(output_schema_example, ensure_ascii=False, indent=2)}

Raw PDF text, secondary aid only:
{raw_text[:12000]}
"""
    return textwrap.dedent(prompt).strip()


def parse_gemini_json_response(response_text: Optional[str]) -> Dict[str, Any]:
    if not response_text:
        raise ValueError("Empty Gemini response text.")

    text = response_text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise


def validate_price_grid_row_dict(row: Dict[str, Any]) -> List[str]:
    errors = []
    if not row.get("device_name"):
        errors.append("missing_device_name")
    if not row.get("price_text") and row.get("price_value") is None:
        errors.append("missing_price")
    if not row.get("offer_bucket"):
        errors.append("missing_offer_bucket")
    if not row.get("raw_evidence"):
        errors.append("missing_raw_evidence")
    return errors


def call_gemini_for_page(page_row: pd.Series, section_row: pd.Series) -> Dict[str, Any]:
    client = make_gemini_client()

    image_path = page_row.get("rendered_image_path")
    image_bytes = read_binary_file(image_path)

    section_id = section_row.get("section_id")
    layout_type = section_row.get("layout_type")
    title_header_text = section_row.get("title_header_text") or ""
    raw_text = page_row.get("raw_text") or ""

    prompt = build_vision_prompt(section_id, layout_type, title_header_text, raw_text)

    image_part = genai_types.Part.from_bytes(
        data=image_bytes,
        mime_type="image/png",
    )

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[prompt, image_part],
        config=genai_types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
        ),
    )

    response_text = getattr(response, "text", None)
    if not response_text and hasattr(response, "output_text"):
        response_text = response.output_text

    parsed_json = parse_gemini_json_response(response_text)

    return {
        "parsed_json": parsed_json,
        "raw_response_text": response_text,
    }


def run_gemini_vision_extraction(
    slide_raw_df: pd.DataFrame,
    slide_sections_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    print_header("GEMINI VISION EXTRACTION")

    table_rows = []
    offer_rows = []
    price_grid_rows = []

    if not USE_GEMINI_VISION:
        print("USE_GEMINI_VISION is False. Skipping Gemini Vision extraction.")
        return pd.DataFrame(table_rows), pd.DataFrame(offer_rows), pd.DataFrame(price_grid_rows)

    vision_pages = slide_sections_df[
        slide_sections_df["layout_type"].isin(VISION_LAYOUT_TYPES)
        & (slide_sections_df["section_id"] != "unknown")
    ].copy()

    if vision_pages.empty:
        print("No pages routed to Gemini Vision.")
        return pd.DataFrame(table_rows), pd.DataFrame(offer_rows), pd.DataFrame(price_grid_rows)

    print(f"Gemini pages queued: {len(vision_pages):,}")

    processed_count = 0
    for _, section_row in vision_pages.sort_values("page_num").iterrows():
        if processed_count >= MAX_GEMINI_PAGES_PER_RUN:
            print(f"Reached MAX_GEMINI_PAGES_PER_RUN={MAX_GEMINI_PAGES_PER_RUN}. Stopping Gemini extraction.")
            break

        page_num = int(section_row["page_num"])
        page_row = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]
        section_id = section_row["section_id"]
        layout_type = section_row["layout_type"]

        print_subheader(f"Gemini Vision page {page_num}: {section_id} | {layout_type}")

        extraction_status = "success"
        error_message = None
        parsed_json = None
        raw_response_text = None

        try:
            result = call_gemini_for_page(page_row, section_row)
            parsed_json = result["parsed_json"]
            raw_response_text = result["raw_response_text"]

            rows = parsed_json.get("rows") or []
            grid_rows = parsed_json.get("price_grid_rows") or []

            print(f"Gemini offer rows returned: {len(rows):,}")
            print(f"Gemini price grid rows returned: {len(grid_rows):,}")

            for idx, row in enumerate(rows):
                validation_errors = validate_offer_candidate_dict(row, section_id)
                confidence = safe_float(row.get("confidence"))
                if confidence is None:
                    confidence = 0.75
                needs_review = bool(row.get("needs_review")) or bool(validation_errors) or confidence < AUTO_APPROVE_MIN_CONFIDENCE

                candidate_id = make_hash_id(
                    "cand",
                    deck_id,
                    page_num,
                    section_id,
                    row.get("carrier"),
                    row.get("product_name"),
                    row.get("offer_type"),
                    row.get("raw_evidence"),
                    idx,
                )
                block_id = make_hash_id("block", deck_id, page_num, section_id, "gemini_vision", idx)

                offer_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "candidate_id": candidate_id,
                    "block_id": block_id,
                    "page_num": page_num,
                    "section_id": section_id,
                    "layout_type": layout_type,
                    "carrier": row.get("carrier"),
                    "brand": row.get("brand"),
                    "product_name": row.get("product_name"),
                    "product_category": row.get("product_category") or row.get("category"),
                    "offer_type": row.get("offer_type"),
                    "offer_amount_text": row.get("offer_amount_text"),
                    "offer_value": safe_float(row.get("offer_value")) if row.get("offer_value") is not None else extract_offer_value(row.get("offer_amount_text")),
                    "currency": row.get("currency") or "USD",
                    "rate_plan_requirement": row.get("rate_plan_requirement"),
                    "customer_segment": row.get("customer_segment"),
                    "action_required": row.get("action_required"),
                    "date_start_text": row.get("date_start_text"),
                    "date_end_text": row.get("date_end_text"),
                    "condition_text": row.get("condition_text"),
                    "raw_evidence": row.get("raw_evidence"),
                    "extraction_method": "gemini_vision",
                    "confidence": confidence,
                    "needs_review": needs_review,
                    "validation_errors_json": safe_json(validation_errors),
                    "created_ts": now_utc(),
                })

            for idx, grid_row in enumerate(grid_rows):
                validation_errors = validate_price_grid_row_dict(grid_row)
                confidence = safe_float(grid_row.get("confidence"))
                if confidence is None:
                    confidence = 0.75
                needs_review = bool(grid_row.get("needs_review")) or bool(validation_errors) or confidence < AUTO_APPROVE_MIN_CONFIDENCE

                grid_row_id = make_hash_id(
                    "grid",
                    deck_id,
                    page_num,
                    section_id,
                    grid_row.get("brand"),
                    grid_row.get("device_name"),
                    grid_row.get("offer_bucket"),
                    grid_row.get("price_text"),
                    idx,
                )

                price_grid_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "grid_row_id": grid_row_id,
                    "page_num": page_num,
                    "section_id": section_id,
                    "brand": grid_row.get("brand"),
                    "device_name": grid_row.get("device_name"),
                    "device_category": grid_row.get("device_category"),
                    "retail_price": safe_float(grid_row.get("retail_price")),
                    "offer_bucket": grid_row.get("offer_bucket"),
                    "customer_scenario": grid_row.get("customer_scenario"),
                    "id_requirement": grid_row.get("id_requirement"),
                    "rate_plan_requirement": grid_row.get("rate_plan_requirement"),
                    "price_text": grid_row.get("price_text"),
                    "price_value": safe_float(grid_row.get("price_value")) if grid_row.get("price_value") is not None else safe_float(grid_row.get("price_text")),
                    "raw_evidence": grid_row.get("raw_evidence"),
                    "extraction_method": "gemini_vision_price_grid",
                    "confidence": confidence,
                    "needs_review": needs_review,
                    "validation_errors_json": safe_json(validation_errors),
                    "created_ts": now_utc(),
                })

        except Exception:
            extraction_status = "error"
            error_message = traceback.format_exc()
            print(f"Gemini failed on page {page_num}.")
            print(error_message)

        table_rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "extractor_name": "gemini_vision_page_extractor",
            "extractor_model": GEMINI_MODEL,
            "prompt_version": PROMPT_VERSION,
            "raw_json": safe_json(parsed_json),
            "raw_response_text": raw_response_text,
            "extraction_status": extraction_status,
            "error_message": error_message,
            "created_ts": now_utc(),
        })

        processed_count += 1
        time.sleep(GEMINI_SLEEP_SECONDS)

    slide_tables_df = pd.DataFrame(table_rows)
    vision_candidates_df = pd.DataFrame(offer_rows)
    price_grid_df = pd.DataFrame(price_grid_rows)

    if not vision_candidates_df.empty:
        vision_candidates_df = vision_candidates_df.drop_duplicates(subset=["candidate_id"])
    if not price_grid_df.empty:
        price_grid_df = price_grid_df.drop_duplicates(subset=["grid_row_id"])

    print(f"Gemini slide table rows: {len(slide_tables_df):,}")
    print(f"Gemini offer candidate rows: {len(vision_candidates_df):,}")
    print(f"Gemini price grid rows: {len(price_grid_df):,}")

    return slide_tables_df, vision_candidates_df, price_grid_df


# ======================================================================================
# SECTION 14 - NORMALIZATION AND DEVICE BRIDGE
# ======================================================================================

def infer_device_brand(device_name: Optional[str]) -> Optional[str]:
    if not device_name:
        return None
    norm = normalize_for_match(device_name)
    if "iphone" in norm or "ipad" in norm or "apple watch" in norm:
        return "Apple"
    if "galaxy" in norm or "samsung" in norm:
        return "Samsung"
    if "pixel" in norm:
        return "Google"
    if "moto" in norm or "motorola" in norm or "razr" in norm:
        return "Motorola"
    if "tcl" in norm:
        return "TCL"
    if "revvl" in norm:
        return "T-Mobile"
    if "gizmo" in norm:
        return "Verizon"
    if "franklin" in norm:
        return "Franklin"
    if "netgear" in norm:
        return "Netgear"
    if "inseego" in norm:
        return "Inseego"
    return None


def infer_product_category(product_name: Optional[str], section_id: Optional[str], raw_evidence: Optional[str]) -> Optional[str]:
    norm = normalize_for_match(" ".join([product_name or "", section_id or "", raw_evidence or ""]))
    if "watch" in norm:
        return "watch"
    if "tablet" in norm or "ipad" in norm or " tab " in f" {norm} ":
        return "tablet"
    if "byod" in norm:
        return "byod"
    if "hint" in norm or "gateway" in norm or "internet" in norm:
        return "home_internet"
    if "hotspot" in norm or "mifi" in norm or "router" in norm:
        return "hotspot_router"
    if "service" in norm or "rate plan" in norm or " plan " in f" {norm} ":
        return "service_plan"
    if "phone" in norm or "iphone" in norm or "galaxy" in norm or "pixel" in norm or "moto" in norm:
        return "smartphone"
    return None


def normalize_offers(
    offer_candidates_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("SILVER NORMALIZED OFFERS")

    if offer_candidates_df is None or offer_candidates_df.empty:
        print("No offer candidates to normalize.")
        return pd.DataFrame()

    rows = []
    for _, candidate in offer_candidates_df.iterrows():
        product_name = candidate.get("product_name")
        product_category = candidate.get("product_category") or infer_product_category(
            product_name,
            candidate.get("section_id"),
            candidate.get("raw_evidence"),
        )
        brand = candidate.get("brand") or infer_device_brand(product_name)
        confidence = safe_float(candidate.get("confidence")) or 0.0
        needs_review = bool(candidate.get("needs_review"))
        offer_type = candidate.get("offer_type")
        auto_approve_flag = bool((not needs_review) and confidence >= AUTO_APPROVE_MIN_CONFIDENCE and offer_type != "no_offer")

        offer_id = make_hash_id(
            "offer",
            candidate.get("candidate_id"),
            candidate.get("carrier"),
            product_name,
            offer_type,
            candidate.get("raw_evidence"),
        )

        rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "offer_id": offer_id,
            "candidate_id": candidate.get("candidate_id"),
            "page_num": int(candidate.get("page_num")),
            "section_id": candidate.get("section_id"),
            "layout_type": candidate.get("layout_type"),
            "carrier": candidate.get("carrier"),
            "brand": brand,
            "product_name": product_name,
            "product_category": product_category,
            "offer_type": offer_type,
            "offer_amount_text": candidate.get("offer_amount_text"),
            "offer_value": safe_float(candidate.get("offer_value")),
            "currency": candidate.get("currency") or "USD",
            "rate_plan_requirement": candidate.get("rate_plan_requirement"),
            "customer_segment": candidate.get("customer_segment"),
            "action_required": candidate.get("action_required"),
            "condition_text": candidate.get("condition_text"),
            "date_start": parse_date_safe(candidate.get("date_start_text")),
            "date_end": parse_date_safe(candidate.get("date_end_text")),
            "raw_evidence": candidate.get("raw_evidence"),
            "normalization_method": "rule_normalizer_v2",
            "confidence": confidence,
            "needs_review": needs_review,
            "auto_approve_flag": auto_approve_flag,
            "created_ts": now_utc(),
        })

    normalized_df = pd.DataFrame(rows)
    if not normalized_df.empty:
        normalized_df = normalized_df.drop_duplicates(subset=["offer_id"])

    print(f"Normalized offer rows: {len(normalized_df):,}")
    if not normalized_df.empty:
        print(f"Auto-approve rows: {int(normalized_df['auto_approve_flag'].sum()):,}")
    return normalized_df


def extract_device_names_from_text(text: Optional[str]) -> List[str]:
    if not text:
        return []

    patterns = [
        r"iPhone\s+\d+[A-Za-z]*(?:\s+Pro\s+Max|\s+Pro|\s+Plus|\s+Air|\s+e)?",
        r"Galaxy\s+S\d+\s*(?:Ultra|FE|Edge|\+)?",
        r"Galaxy\s+Z\s*(?:Flip|Fold)\d*",
        r"Galaxy\s+A\d+\s*5G?",
        r"Galaxy\s+Tab\s+[A-Za-z0-9+\s]+",
        r"Pixel\s+\d+[A-Za-z]*(?:\s+Pro\s+XL|\s+Pro|\s+Fold)?",
        r"moto\s+g\s+stylus\s+5G\s+\d{4}",
        r"moto\s+g\s+power\s+5G\s+\d{4}",
        r"moto\s+g\s+5G\s+\d{4}",
        r"moto\s+razr\s+\d{4}",
        r"Razr\+?",
        r"Apple\s+Watch\s+[A-Za-z0-9\s]+",
        r"iPad\s+?",
        r"TCL\s+[A-Za-z0-9\s]+",
        r"REVVL\s+[A-Za-z0-9\s]+",
        r"SYNCUP\s+[A-Za-z0-9\s]+",
        r"GizmoWatch\s+\d*",
        r"Inseego\s+MiFi\s+[A-Za-z0-9\s]+",
        r"Netgear\s+Nighthawk\s+[A-Za-z0-9\s]+",
        r"Franklin\s+[A-Za-z0-9\s]+",
    ]

    found = []
    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            value = clean_text_line(match.group(0))
            if 2 <= len(value) <= 100:
                found.append(value)

    deduped = []
    seen = set()
    for value in found:
        key = normalize_for_match(value)
        if key and key not in seen:
            deduped.append(value)
            seen.add(key)
    return deduped


def build_offer_device_bridge(
    normalized_offers_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> pd.DataFrame:
    print_header("SILVER OFFER DEVICE BRIDGE")

    if normalized_offers_df is None or normalized_offers_df.empty:
        print("No normalized offers available for device bridge.")
        return pd.DataFrame()

    rows = []
    for _, offer in normalized_offers_df.iterrows():
        text = " ".join([str(offer.get("product_name") or ""), str(offer.get("raw_evidence") or "")])
        devices = []
        if offer.get("product_name"):
            devices.append(offer.get("product_name"))
        devices.extend(extract_device_names_from_text(text))

        deduped_devices = []
        seen = set()
        for device in devices:
            key = normalize_for_match(device)
            if key and key not in seen:
                deduped_devices.append(device)
                seen.add(key)

        for device in deduped_devices:
            bridge_id = make_hash_id("bridge", offer.get("offer_id"), device)
            rows.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "bridge_id": bridge_id,
                "offer_id": offer.get("offer_id"),
                "candidate_id": offer.get("candidate_id"),
                "page_num": int(offer.get("page_num")),
                "section_id": offer.get("section_id"),
                "carrier": offer.get("carrier"),
                "device_name": device,
                "device_brand": infer_device_brand(device),
                "device_family": infer_product_category(device, offer.get("section_id"), offer.get("raw_evidence")),
                "device_model": device,
                "raw_evidence": offer.get("raw_evidence"),
                "confidence": offer.get("confidence"),
                "created_ts": now_utc(),
            })

    bridge_df = pd.DataFrame(rows)
    if not bridge_df.empty:
        bridge_df = bridge_df.drop_duplicates(subset=["bridge_id"])

    print(f"Offer-device bridge rows: {len(bridge_df):,}")
    return bridge_df


# ======================================================================================
# SECTION 15 - REVIEW QUEUE AND REVIEW DECISIONS
# ======================================================================================

def build_review_tables(
    offer_candidates_df: pd.DataFrame,
    price_grid_rows_df: pd.DataFrame,
    normalized_offers_df: pd.DataFrame,
    deck_id: str,
    run_id: str,
    pdf_hash: str,
    pdf_name: str,
    deck_week: date,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print_header("REVIEW QUEUE AND AUTO-APPROVED DECISIONS")

    review_rows = []
    decision_rows = []

    if offer_candidates_df is not None and not offer_candidates_df.empty:
        for _, candidate in offer_candidates_df.iterrows():
            if bool(candidate.get("needs_review")):
                review_id = make_hash_id("review", "candidate", candidate.get("candidate_id"))
                review_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "review_id": review_id,
                    "candidate_id": candidate.get("candidate_id"),
                    "grid_row_id": None,
                    "page_num": int(candidate.get("page_num")),
                    "section_id": candidate.get("section_id"),
                    "review_reason": candidate.get("validation_errors_json") or "needs_review_candidate",
                    "review_status": "pending",
                    "source_table": "silver_offerCandidates",
                    "raw_payload_json": safe_json(candidate.to_dict()),
                    "created_ts": now_utc(),
                })

    if price_grid_rows_df is not None and not price_grid_rows_df.empty:
        for _, grid_row in price_grid_rows_df.iterrows():
            if bool(grid_row.get("needs_review")):
                review_id = make_hash_id("review", "grid", grid_row.get("grid_row_id"))
                review_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "review_id": review_id,
                    "candidate_id": None,
                    "grid_row_id": grid_row.get("grid_row_id"),
                    "page_num": int(grid_row.get("page_num")),
                    "section_id": grid_row.get("section_id"),
                    "review_reason": grid_row.get("validation_errors_json") or "needs_review_price_grid_row",
                    "review_status": "pending",
                    "source_table": "silver_priceGridRows",
                    "raw_payload_json": safe_json(grid_row.to_dict()),
                    "created_ts": now_utc(),
                })

    if normalized_offers_df is not None and not normalized_offers_df.empty:
        auto_df = normalized_offers_df[normalized_offers_df["auto_approve_flag"] == True].copy()
        for _, offer in auto_df.iterrows():
            decision_id = make_hash_id("decision", "auto", offer.get("candidate_id"))
            decision_rows.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "decision_id": decision_id,
                "candidate_id": offer.get("candidate_id"),
                "grid_row_id": None,
                "page_num": int(offer.get("page_num")),
                "section_id": offer.get("section_id"),
                "decision_type": "auto_approved",
                "decision_status": "approved",
                "decision_source": "system_rules_v2",
                "decision_notes": f"confidence={offer.get('confidence')}; threshold={AUTO_APPROVE_MIN_CONFIDENCE}",
                "created_ts": now_utc(),
            })

    review_df = pd.DataFrame(review_rows)
    decisions_df = pd.DataFrame(decision_rows)

    print(f"Review queue rows: {len(review_df):,}")
    print(f"Auto-approved decision rows: {len(decisions_df):,}")

    return review_df, decisions_df


# ======================================================================================
# SECTION 16 - PIPELINE RUNNER
# ======================================================================================

def run_pipeline() -> Dict[str, Any]:
    print_header("STARTING COMPETITIVE OFFER REPORT PIPELINE V2")

    if WRITE_TO_BQ:
        ensure_dataset_and_tables()

    pdf_path = choose_pdf_path()
    pdf_name = os.path.basename(pdf_path)
    pdf_hash = compute_file_hash(pdf_path)
    file_size_bytes = os.path.getsize(pdf_path)
    run_id = f"run_{uuid.uuid4().hex[:16]}"

    temp_doc = fitz.open(pdf_path)
    page_count = temp_doc.page_count
    first_page_text = temp_doc[0].get_text("text") if page_count > 0 else ""
    temp_doc.close()

    deck_week = infer_deck_week_from_text_or_name(pdf_path, first_page_text)
    deck_id = f"deck_{deck_week.strftime('%Y%m%d')}_{pdf_hash[:12]}"

    print(f"pdf_name: {pdf_name}")
    print(f"pdf_hash: {pdf_hash}")
    print(f"deck_week: {deck_week}")
    print(f"deck_id: {deck_id}")
    print(f"run_id: {run_id}")
    print(f"page_count: {page_count}")

    duplicate_action = "new"
    existing_df = get_existing_decks_by_hash(pdf_hash)
    if not existing_df.empty:
        duplicate_action = resolve_duplicate_action(existing_df)
        if duplicate_action == "replace":
            delete_existing_rows_for_pdf_hash(pdf_hash)

    pdf_decks_df = pd.DataFrame([{
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "pdf_name": pdf_name,
        "deck_week": deck_week,
        "pdf_path": pdf_path,
        "page_count": page_count,
        "file_size_bytes": file_size_bytes,
        "duplicate_action": duplicate_action,
        "pipeline_version": PIPELINE_VERSION,
        "ingestion_ts": now_utc(),
    }])

    slide_raw_df, slide_lines_df = extract_pdf_to_bronze(
        pdf_path=pdf_path,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        deck_week=deck_week,
    )

    detected_entities_df = detect_entities(
        slide_raw_df=slide_raw_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    toc_map_df = parse_toc_page_map(
        slide_raw_df=slide_raw_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    slide_sections_df = classify_all_slides(
        slide_raw_df=slide_raw_df,
        slide_lines_df=slide_lines_df,
        toc_map_df=toc_map_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    content_blocks_df = build_content_blocks(
        slide_raw_df=slide_raw_df,
        slide_sections_df=slide_sections_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    deterministic_candidates_df = build_offer_candidates_from_blocks(
        content_blocks_df=content_blocks_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    slide_tables_df, vision_candidates_df, price_grid_rows_df = run_gemini_vision_extraction(
        slide_raw_df=slide_raw_df,
        slide_sections_df=slide_sections_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    candidate_frames = []
    if deterministic_candidates_df is not None and not deterministic_candidates_df.empty:
        candidate_frames.append(deterministic_candidates_df)
    if vision_candidates_df is not None and not vision_candidates_df.empty:
        candidate_frames.append(vision_candidates_df)

    if candidate_frames:
        offer_candidates_df = pd.concat(candidate_frames, ignore_index=True)
        offer_candidates_df = offer_candidates_df.drop_duplicates(subset=["candidate_id"])
    else:
        offer_candidates_df = pd.DataFrame()

    print_header("COMBINED OFFER CANDIDATES")
    print(f"Combined offer candidate rows: {len(offer_candidates_df):,}")

    normalized_offers_df = normalize_offers(
        offer_candidates_df=offer_candidates_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    offer_device_bridge_df = build_offer_device_bridge(
        normalized_offers_df=normalized_offers_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    review_df, decisions_df = build_review_tables(
        offer_candidates_df=offer_candidates_df,
        price_grid_rows_df=price_grid_rows_df,
        normalized_offers_df=normalized_offers_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    if WRITE_TO_BQ:
        print_header("BIGQUERY LOAD")
        append_df_to_bq(pdf_decks_df, "bronze_pdfDecks")
        append_df_to_bq(slide_raw_df, "bronze_slideRaw")
        append_df_to_bq(slide_lines_df, "bronze_slideLines")
        append_df_to_bq(detected_entities_df, "bronze_detectedEntities")
        append_df_to_bq(slide_tables_df, "bronze_slideTables")

        append_df_to_bq(toc_map_df, "silver_tocPageMap")
        append_df_to_bq(slide_sections_df, "silver_slideSections")
        append_df_to_bq(content_blocks_df, "silver_slideContentBlocks")
        append_df_to_bq(offer_candidates_df, "silver_offerCandidates")
        append_df_to_bq(normalized_offers_df, "silver_normalizedOffers")
        append_df_to_bq(offer_device_bridge_df, "silver_offerDeviceBridge")
        append_df_to_bq(price_grid_rows_df, "silver_priceGridRows")

        append_df_to_bq(review_df, "review_extractionReview")
        append_df_to_bq(decisions_df, "review_reviewDecisions")

    summary = {
        "pdf_name": pdf_name,
        "deck_week": str(deck_week),
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "duplicate_action": duplicate_action,
        "page_count": page_count,
        "bronze_slide_raw_rows": len(slide_raw_df),
        "bronze_slide_line_rows": len(slide_lines_df),
        "detected_entity_rows": len(detected_entities_df),
        "toc_map_rows": len(toc_map_df),
        "slide_section_rows": len(slide_sections_df),
        "content_block_rows": len(content_blocks_df),
        "slide_table_rows": len(slide_tables_df),
        "offer_candidate_rows": len(offer_candidates_df),
        "normalized_offer_rows": len(normalized_offers_df),
        "offer_device_bridge_rows": len(offer_device_bridge_df),
        "price_grid_rows": len(price_grid_rows_df),
        "review_queue_rows": len(review_df),
        "auto_approved_decision_rows": len(decisions_df),
    }

    print_header("FINAL RUN SUMMARY")
    for key, value in summary.items():
        print(f"{key:<35}: {value}")

    if PRINT_SAMPLE_OUTPUTS:
        print_header("SAMPLE OUTPUTS")

        print_subheader("Slide sections")
        show_df(slide_sections_df[[
            "page_num",
            "section_id",
            "layout_type",
            "classifier_method",
            "toc_expected_section",
            "toc_mismatch_flag",
            "title_header_text",
        ]].sort_values("page_num"), max_rows=200)

        if not offer_candidates_df.empty:
            print_subheader("Offer candidates sample")
            sample_cols = [
                "page_num",
                "section_id",
                "carrier",
                "product_name",
                "offer_type",
                "offer_amount_text",
                "confidence",
                "needs_review",
                "raw_evidence",
            ]
            show_df(offer_candidates_df[sample_cols], max_rows=50)

        if not price_grid_rows_df.empty:
            print_subheader("Price grid sample")
            show_df(price_grid_rows_df, max_rows=50)

        if not review_df.empty:
            print_subheader("Review queue sample")
            show_df(review_df[["page_num", "section_id", "review_reason", "source_table"]], max_rows=50)

    print_header("PIPELINE COMPLETE")

    return {
        "summary": summary,
        "pdf_decks_df": pdf_decks_df,
        "slide_raw_df": slide_raw_df,
        "slide_lines_df": slide_lines_df,
        "detected_entities_df": detected_entities_df,
        "toc_map_df": toc_map_df,
        "slide_sections_df": slide_sections_df,
        "content_blocks_df": content_blocks_df,
        "slide_tables_df": slide_tables_df,
        "offer_candidates_df": offer_candidates_df,
        "normalized_offers_df": normalized_offers_df,
        "offer_device_bridge_df": offer_device_bridge_df,
        "price_grid_rows_df": price_grid_rows_df,
        "review_df": review_df,
        "decisions_df": decisions_df,
    }


# ======================================================================================
# SECTION 17 - EXECUTE PIPELINE
# ======================================================================================

pipeline_outputs = run_pipeline()

     